# Esfera viscoelástica: energia interna, mapa de estados ressonantes e relações de existência e de recorrência

Notebook ilustrativo do capítulo *Centro espalhador elástico* da tese: esfera elástica e modelos viscoelásticos de
Kelvin–Voigt e de Maxwell. Cada parte apresenta primeiro as equações (na mesma notação do texto) e depois o código que as
calcula e as figuras correspondentes.

1. Configurações e estética das figuras
2. Modelo: potenciais, condições de contorno, coeficientes e velocidades complexas
3. Energia interna: fórmula de superfície e verificação numérica
4. Casos-limite: esfera fluida ($c_T\to0$) e esfera elástica ($\tilde\tau_V\to0$ e $\tilde\tau_M\to\infty$)
5. Curvas de energia: elástico, Kelvin–Voigt e Maxwell
6. Banco de dados e contagem das ressonâncias
7. Mapa de estados ressonantes
8. Relação de existência (caso elástico)
9. Relação de recorrência (caso elástico)
10. Custo computacional
11. Arquivos gerados

As partes 1 a 5 só precisam deste notebook. As partes 6 a 10 leem a saída do gerador da esfera sólida
(`gerador_viscoelastico.py`, versão 2), uma pasta por modelo (`viscoelastico_lmax10_elastico`, `viscoelastico_lmax10_voigt` e
`viscoelastico_lmax10_maxwell`), cada uma com `resumo_viscoelastico_lmax10_*.csv`, `hipercubo_*.npz` e
`parametros_viscoelastico_lmax10_*.json`, dentro da pasta indicada em `PASTA`.

## 1. Configurações
No Colab, a célula abaixo instala o LaTeX usado nos rótulos (só na primeira vez). No PC, basta ter uma distribuição LaTeX com `dvipng` e `cm-super`.

**Onde ficam os dados e os resultados.** No Colab, o Google Drive é montado logo no início; os dados são lidos de
`Meu Drive/visco_final/` e todas as figuras e tabelas são gravadas em `Meu Drive/visco_final/resultados/`, ao lado dos dados,
de modo que nada se perde quando a sessão termina. Fora do Colab, a pasta dos dados vem da variável de ambiente `VISCO_DADOS`
(se ela não existir, usa-se `Downloads/visco_final`), e os resultados vão para `resultados/`, na pasta do notebook, ou para a pasta
indicada em `VISCO_SAIDA`:

```
resultados/
├── figuras/   png (1200 dpi), pdf e eps de cada figura
└── tabelas/   csv, tex, png, pdf e svg de cada tabela
```

In [ ]:
%%capture
import os, sys, subprocess
if os.path.exists("/content"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

In [ ]:
import os, glob, time, shutil, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap, Normalize, LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, NullFormatter, FuncFormatter
from scipy.special import spherical_jn, spherical_yn
from scipy.integrate import quad
from scipy.spatial import cKDTree
from scipy.signal import find_peaks
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

USA_LATEX = shutil.which("latex") is not None
ESTILO_RC = {
    "text.usetex": USA_LATEX,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "figure.dpi": 150,                      # só a exibição no notebook; os arquivos saem com OUTPUT_DPI
}
plt.rcParams.update(ESTILO_RC)
VIRG = "{,}" if USA_LATEX else ","          # vírgula decimal ("{,}" evita o espaço extra do LaTeX)
PCT = r"\%" if USA_LATEX else "%"

# estética padrão das figuras da tese
OUTPUT_DPI = 1200
COLORS = ["#6488A5", "#8B1A1A", "black", "0.45"]
LINESTYLES = ["-", "--", "-.", ":"]
LINEWIDTHS = [2.0, 2.0, 1.75, 1.6]

# ---------------- pastas: dados e resultados ----------------
# Google Drive: Meu Drive > visco_final (uma subpasta por modelo, saída do gerador)
try:
    from google.colab import drive
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/visco_final"                      # saída do gerador
    SAIDA = os.path.join(PASTA, "resultados")                          # figuras e tabelas, gravadas no Drive
else:
    PASTA = os.path.abspath(os.environ.get("VISCO_DADOS", os.path.join(os.path.expanduser("~"), "Downloads", "visco_final")))
    SAIDA = os.path.abspath(os.environ.get("VISCO_SAIDA", "resultados"))
FIG_DIR = os.path.join(SAIDA, "figuras")
TAB_DIR = os.path.join(SAIDA, "tabelas")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def salvar_figura(fig, nome):
    base = os.path.join(FIG_DIR, nome)
    fig.savefig(base + ".png", dpi=OUTPUT_DPI, bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")
    fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")
    fig.savefig(base + ".eps", format="eps", bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")


def dec(valor, casas):
    return f"{valor:.{casas}f}".replace(".", VIRG)


def sci(valor, casas=1):
    # notação científica com vírgula decimal, para o modo matemático: 3{,}2\times10^{-9}
    if not np.isfinite(valor):
        return "--"
    if valor == 0:
        return "0"
    e = int(np.floor(np.log10(abs(valor))))
    m = valor / 10.0 ** e
    if round(abs(m), casas) >= 10:
        m, e = m / 10, e + 1
    if casas == 0 and round(abs(m)) == 1:
        return rf"{'-' if m < 0 else ''}10^{{{e}}}"
    return rf"{dec(m, casas)}\times10^{{{e}}}"


def milhar(n):
    # espaço fino nos milhares (ABNT): 8\,000
    return f"{int(n):,}".replace(",", r"\,")


def estilizar(ax, xlabel=None, ylabel=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=12)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(axis="both", which="both", direction="in", length=4, width=0.9, labelsize=10)
    ax.tick_params(axis="both", which="minor", length=2)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_edgecolor("black")
    ax.grid(False)


def adicionar_caixa(ax, texto, x=0.965, y=0.955, ha="right"):
    ax.text(x, y, texto, transform=ax.transAxes, ha=ha, va="top", fontsize=9, linespacing=1.10, zorder=10,
            bbox={"boxstyle": "square,pad=0.30", "facecolor": "white", "edgecolor": "black", "linewidth": 1.0, "alpha": 1.0})


def adicionar_legenda(ax, loc="lower right", **kw):
    leg = ax.legend(loc=loc, fontsize=kw.pop("fontsize", 9), frameon=True, handlelength=2.8, labelspacing=0.30,
                    handletextpad=0.7, **kw)
    leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.0, alpha=1.0)
    return leg


# ---------------- tabelas (mesmo formato das tabelas da tese) ----------------
def fmt_pm(valor, erro):
    if not (np.isfinite(erro) and erro > 0.0):
        return f"${dec(valor, 4)}$"
    casas = max(0, -int(np.floor(np.log10(erro))) + 1)
    return rf"${f'{valor:+.{casas}f}'.replace('.', VIRG)}\pm{f'{erro:.{casas}f}'.replace('.', VIRG)}$"


def save_compact_table(base, dados, linhas, colunas, figsize, first_col_width=0.16, header_label="Coef.", fontsize=17):
    with plt.rc_context({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 30}):
        fig, ax = plt.subplots(figsize=figsize, facecolor="white")
        ax.axis("off")
        tab = ax.table(cellText=[[linhas[i], *dados[i]] for i in range(len(linhas))], colLabels=[header_label, *colunas],
                       cellLoc="center", colWidths=[first_col_width] + [(1 - first_col_width) / len(colunas)] * len(colunas),
                       bbox=[0.02, 0.02, 0.96, 0.96])
        tab.auto_set_font_size(False)
        tab.set_fontsize(fontsize)
        for (lin, col), cel in tab.get_celld().items():
            cel.visible_edges = ""
            cel.set_linewidth(0.0)
            cel.PAD = 0.01
            cel.get_text().set_ha("left" if col == 0 else "center")
            cel.get_text().set_linespacing(0.90)
        for col in range(len(colunas) + 1):
            tab[(0, col)].visible_edges = "TB"
            tab[(0, col)].set_linewidth(1.5)
            tab[(0, col)].get_text().set_fontsize(fontsize + 1)
            tab[(len(linhas), col)].visible_edges = "B"
            tab[(len(linhas), col)].set_linewidth(1.5)
        tab[(0, 0)].get_text().set_fontweight("bold")
        fig.subplots_adjust(left=0.005, right=0.995, bottom=0.005, top=0.995)
        for ext in ("png", "pdf", "svg"):
            fig.savefig(f"{base}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.01, facecolor="white")
        plt.show()
        plt.close(fig)


def salvar_tabela_coeficientes(nome, linhas, colunas, valores, erros, figsize, first_col_width=0.16, fontsize=17):
    base = os.path.join(TAB_DIR, nome)
    dados = [[fmt_pm(valores[i][j], erros[i][j]) for j in range(len(colunas))] for i in range(len(linhas))]
    save_compact_table(base, dados, linhas, colunas, figsize, first_col_width, fontsize=fontsize)
    pd.DataFrame([{"coeficiente": l.replace("$", ""), "coluna": c.replace("\n", " "), "valor": valores[i][j], "erro_padrao": erros[i][j]}
                  for i, l in enumerate(linhas) for j, c in enumerate(colunas)]).to_csv(base + ".csv", index=False)
    with open(base + ".tex", "w", encoding="utf-8") as fh:
        fh.write("\\begin{tabular}{l" + "c" * len(colunas) + "}\n\\hline\n")
        fh.write("Coef. & " + " & ".join(c.replace("\n", " ") for c in colunas) + " \\\\\n\\hline\n")
        for i, l in enumerate(linhas):
            fh.write(l + " & " + " & ".join(dados[i]) + " \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")


def salvar_tabela_texto(nome, linhas, colunas, dados, valores, figsize, first_col_width=0.30, header_label="", fontsize=15):
    # tabela de valores já formatados (dados, em LaTeX) + csv com os valores numéricos (valores, DataFrame)
    base = os.path.join(TAB_DIR, nome)
    save_compact_table(base, dados, linhas, colunas, figsize, first_col_width, header_label=header_label, fontsize=fontsize)
    valores.to_csv(base + ".csv")
    with open(base + ".tex", "w", encoding="utf-8") as fh:
        fh.write("\\begin{tabular}{l" + "c" * len(colunas) + "}\n\\hline\n")
        fh.write(header_label + " & " + " & ".join(c.replace("\n", " ") for c in colunas) + " \\\\\n\\hline\n")
        for i, l in enumerate(linhas):
            fh.write(l + " & " + " & ".join(dados[i]) + " \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")


# ---------------- elementos visuais comuns a todas as figuras ----------------
BOX_COLOR = "#600B00"
RESONANCE_CMAP = LinearSegmentedColormap.from_list("resonance_count", ["white", "#D9E1EA", "#526F92", "black"], N=256)
RESONANCE_CMAP.set_bad("#D9D9D9")
FRONT_OBS = dict(color="#7FA6D9", linestyle=(0, (5.0, 2.2)), linewidth=2.0, alpha=0.95)
FRONT_LOG = dict(color="#600B00", linestyle=(0, (4.5, 2.5)), linewidth=1.9, alpha=0.92)
MARCADOR_DADOS = dict(s=10, facecolors="white", edgecolors="0.45", linewidths=0.5)


def rotulo_x(v, casas):
    return r"$0$" if np.isclose(v, 0.0) else f"${dec(v, casas)}$"


def fmt_sig(valor, sig=2):
    # valor com vírgula decimal; notação científica fora de [0,01; 100)
    if np.isclose(valor, 1.0):
        return "1"
    if abs(valor) >= 100 or 0 < abs(valor) < 0.01:
        return sci(valor, max(sig - 1, 0))
    return f"{valor:.{sig}g}".replace(".", VIRG)


def eixo_energia(ax, xlim, xticks, ylim, yticks, casas=2, ylabel=r"Energia interna $(u.a.)$"):
    ax.set_yscale("log")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.xaxis.set_major_locator(FixedLocator(xticks))
    ax.set_xticklabels([rotulo_x(v, casas) for v in xticks])
    ax.yaxis.set_major_locator(FixedLocator(yticks))
    ax.set_yticklabels([rf"$10^{{{int(round(np.log10(v)))}}}$" for v in yticks])
    ax.minorticks_off()
    ax.yaxis.set_minor_formatter(NullFormatter())
    estilizar(ax, r"$x_M=k_Ma$", ylabel)


def plotar_curvas(ax, x, curvas, rotulos, picos=False):
    # curvas na ordem COLORS/LINESTYLES/LINEWIDTHS; picos=True marca os máximos com proeminência >= 0,1 em log10 W_T
    n_picos = []
    for i, (W, rot) in enumerate(zip(curvas, rotulos)):
        ok = np.isfinite(W) & (W > 0)
        ax.plot(x[ok], W[ok], color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], label=rot, zorder=2 + i)
        if picos and ok.sum() > 2:
            k = find_peaks(np.log10(W[ok]), prominence=0.1)[0]
            ax.scatter(x[ok][k], W[ok][k], s=12, facecolors="white", edgecolors=COLORS[i], linewidths=0.7, zorder=10)
            n_picos.append(len(k))
    return n_picos


def figura_curvas(x, curvas, rotulos, nome, xlim, xticks, ylim, yticks, casas=2, caixa=None, picos=False, loc="lower right"):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    n_picos = plotar_curvas(ax, x, curvas, rotulos, picos)
    eixo_energia(ax, xlim, xticks, ylim, yticks, casas)
    if caixa:
        adicionar_caixa(ax, caixa, x=0.035, ha="left")
    adicionar_legenda(ax, loc=loc)
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.13, top=0.98)
    salvar_figura(fig, nome)
    plt.show()
    plt.close(fig)
    return n_picos


def hexbin_padrao(ax, x, y, C=None, rotulo_cor="", gridsize=40):
    # hexbin em escala log-log com a reta 1:1; C=None: contagem; senão, mediana de C em cada hexágono
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if C is not None:
        C = np.asarray(C, float)
        ok &= np.isfinite(C)
    if ok.sum() == 0:
        return 0
    lo = 0.8 * min(np.quantile(x[ok], 0.005), np.quantile(y[ok], 0.005))
    hi = 1.25 * max(np.quantile(x[ok], 0.995), np.quantile(y[ok], 0.995))
    ok &= (x >= lo) & (x <= hi) & (y >= lo) & (y <= hi)
    if C is None:
        norm, Cv, red = LogNorm(vmin=1), None, None
    else:
        Cv = np.clip(C[ok], 1e-4, None)
        norm, red = LogNorm(vmin=Cv.min(), vmax=max(Cv.max(), 1.001 * Cv.min())), np.median
    hb = ax.hexbin(x[ok], y[ok], C=Cv, reduce_C_function=red if red else np.mean, xscale="log", yscale="log", gridsize=gridsize,
                   cmap=RESONANCE_CMAP, norm=norm, mincnt=1, linewidths=0.2, edgecolors="white",
                   extent=(np.log10(lo), np.log10(hi), np.log10(lo), np.log10(hi)), zorder=2)
    ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1.0, zorder=3)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    for eixo in (ax.xaxis, ax.yaxis):
        eixo.set_minor_formatter(NullFormatter())
    cb = ax.figure.colorbar(hb, ax=ax, pad=0.02)
    cb.set_label(rotulo_cor, fontsize=10, rotation=270, labelpad=14)
    cb.ax.tick_params(labelsize=9, direction="in")
    cb.ax.yaxis.set_minor_formatter(NullFormatter())
    return int(ok.sum())


print("LaTeX:", USA_LATEX, "| Colab:", NO_COLAB, "\ndados:", PASTA, "\nfiguras:", FIG_DIR, "\ntabelas:", TAB_DIR)

## 2. Modelo

Uma onda plana monocromática incide ao longo de $+z$ sobre uma esfera sólida homogênea e isotrópica de raio $a$, densidade
$\rho_S$ e velocidades longitudinal $c_L$ e transversal $c_T$, imersa em um fluido de densidade $\rho_M$ e velocidade do som $c_M$.
Pela decomposição de Helmholtz, $\mathbf u=-\nabla\psi+\nabla\times\mathbf A$, e com simetria axial,

$$
\begin{aligned}
\psi(r,\theta) &= B\sum_{\ell=0}^{\infty}a_\ell(2\ell+1)\,i^\ell j_\ell(k_Lr)P_\ell(\cos\theta),\\
A_\phi(r,\theta) &= B\sum_{\ell=0}^{\infty}b_\ell(2\ell+1)\,i^\ell j_\ell(k_Tr)\frac{d}{d\theta}P_\ell(\cos\theta),\\
\varphi_M(r,\theta) &= A\sum_{\ell=0}^{\infty}(2\ell+1)\,i^\ell\Big[j_\ell(k_Mr)+s_\ell\,h^{(1)}_\ell(k_Mr)\Big]P_\ell(\cos\theta).
\end{aligned}
$$

Em $r=a$ impõem-se a continuidade da velocidade normal, o equilíbrio da tração normal, $\sigma_{rr}=-p_M$, e a ausência de tensão de
cisalhamento, $\sigma_{r\theta}=0$. Com

$$
x_M=k_Ma\in\mathbb R,\qquad z_L=k_La\in\mathbb C,\qquad z_T=k_Ta\in\mathbb C,\qquad \eta=\frac{c_L}{c_T}=\frac{z_T}{z_L},
$$

as três condições formam, para cada $\ell$, o sistema

$$
\begin{pmatrix}
z_Lj_\ell'(z_L) & \ell(\ell+1)j_\ell(z_T) & -x_Mh_\ell^{(1)\prime}(x_M)\\
z_L^2\big[\lambda j_\ell(z_L)-2\mu j_\ell''(z_L)\big] & -2\mu\,\ell(\ell+1)\big[z_Tj_\ell'(z_T)-j_\ell(z_T)\big] & -\rho_Mc_M^2x_M^2h_\ell^{(1)}(x_M)\\
2\big[z_Lj_\ell'(z_L)-j_\ell(z_L)\big] & z_T^2j_\ell''(z_T)+\big[\ell(\ell+1)-2\big]j_\ell(z_T) & 0
\end{pmatrix}
\begin{pmatrix}a_\ell\\ b_\ell\\ s_\ell\end{pmatrix}
=
\begin{pmatrix}x_Mj_\ell'(x_M)\\ \rho_Mc_M^2x_M^2j_\ell(x_M)\\ 0\end{pmatrix},
$$

com $\mu=\rho_Sc_T^2$ e $\lambda=\rho_Sc_L^2-2\mu$. Para $\ell=0$ não há campo transversal ($b_0=0$) e restam as duas primeiras linhas.
Pelo princípio da correspondência, o caso viscoelástico usa o mesmo sistema com $c_L$ e $c_T$ complexos e dependentes de $x_M$.
Só o cisalhamento dissipa ($K$ real), e a perda é controlada por um único parâmetro adimensional:

- **Kelvin–Voigt**, $\tilde\tau_V=\tau_Tc_M/a$: $\tan\delta_T=\tilde\tau_Vx_M$, $\tan\delta_L=\dfrac{4}{3\eta_0^2}\tan\delta_T$ e $c_j=c_{j,0}\sqrt{1-i\tan\delta_j}$, $j=L,T$;
- **Maxwell**, $\tilde\tau_M=\tau_Tc_M/a$: $\tan\delta_T=1/(\tilde\tau_Mx_M)$, $c_T=\dfrac{c_{T,\infty}}{\sqrt{1+i\tan\delta_T}}$ e
  $c_L=c_{L,\infty}\sqrt{1-\dfrac{4}{3\eta_\infty^2}\left[1-\dfrac{1}{1+i\tan\delta_T}\right]}$;
- **elástico**: $c_L$ e $c_T$ reais e constantes (limite $\tilde\tau_V\to0$ do Kelvin–Voigt e $\tilde\tau_M\to\infty$ do Maxwell).

O meio externo é o do gerador, água com $c_M=1500$ m/s e $\rho_M=1000$ kg/m$^3$. A célula abaixo resolve o sistema diretamente,
sem a forma fechada de $a_\ell$, e confere o wronskiano $j_\ell h_\ell^{(1)\prime}-j_\ell'h_\ell^{(1)}=i/x^2$ e o limite fluido de $a_\ell$,

$$
a_\ell\xrightarrow[\;c_T\to0\;]{}\frac{i\rho_M}{x_M^2\rho_S}\,\frac{1}{j_\ell(z_L)h_\ell^{(1)\prime}(x_M)-(Z_M/Z_S)\,j_\ell'(z_L)h_\ell^{(1)}(x_M)},\qquad Z_M=\rho_Mc_M,\quad Z_S=\rho_Sc_L .
$$

In [ ]:
CM, RHOM = 1500.0, 1000.0          # meio externo do gerador (água)
L_MAX = 10                          # l_max do banco de dados


def j_esf(l, z):
    return spherical_jn(l, z)


def dj_esf(l, z):
    return spherical_jn(l, z, derivative=True)


def ddj_esf(l, z):
    # equação de Bessel esférica: z^2 j'' + 2 z j' + [z^2 - l(l+1)] j = 0
    return -(2.0 / z) * dj_esf(l, z) - (1.0 - l * (l + 1) / z ** 2) * j_esf(l, z)


def h_esf(l, x):
    return spherical_jn(l, x) + 1j * spherical_yn(l, x)


def dh_esf(l, x):
    return spherical_jn(l, x, derivative=True) + 1j * spherical_yn(l, x, derivative=True)


def velocidades(xM, modelo, cL0, cT0, tau=0.0):
    # elástico: c_L, c_T reais; Kelvin-Voigt: c_j = c_j0 sqrt(1 - i tan d_j); Maxwell: c_T e c_L do limite de alta frequência
    x = np.asarray(xM, dtype=float)
    eta2 = (cL0 / cT0) ** 2
    if modelo == "elastico":
        return cL0 * np.ones_like(x, dtype=complex), cT0 * np.ones_like(x, dtype=complex)
    if modelo == "voigt":
        tanT = tau * x
        return cL0 * np.sqrt(1 - 1j * 4 / (3 * eta2) * tanT), cT0 * np.sqrt(1 - 1j * tanT)
    if modelo == "maxwell":
        tanT = 1.0 / (tau * x)
        return cL0 * np.sqrt(1 - 4 / (3 * eta2) * (1.0 - 1.0 / (1.0 + 1j * tanT))), cT0 / np.sqrt(1 + 1j * tanT)
    raise ValueError(modelo)


def coeficientes(xM, l, cL, cT, rhoS, cM=CM, rhoM=RHOM):
    # resolve [M]_l (a_l, b_l, s_l) = (x j'(x), rho_M c_M^2 x^2 j(x), 0); a segunda linha é dividida por rho_M c_M^2
    x = np.asarray(xM, dtype=float)
    cL, cT = np.broadcast_to(cL, x.shape).astype(complex), np.broadcast_to(cT, x.shape).astype(complex)
    zL, zT = x * cM / cL, x * cM / cT
    mu = rhoS * cT ** 2
    lam = rhoS * cL ** 2 - 2 * mu
    rc = rhoM * cM ** 2
    L = l * (l + 1)
    jL, djL, ddjL = j_esf(l, zL), dj_esf(l, zL), ddj_esf(l, zL)
    h, dh = h_esf(l, x), dh_esf(l, x)
    PA = zL ** 2 * (lam * jL - 2 * mu * ddjL)
    if l == 0:                                  # sem campo transversal: b_0 = 0
        M = np.stack([np.stack([zL * djL, -x * dh], -1), np.stack([PA / rc, -x ** 2 * h], -1)], -2)
        rhs = np.stack([x * dj_esf(0, x), x ** 2 * j_esf(0, x)], -1).astype(complex)
        a, s = np.moveaxis(np.linalg.solve(M, rhs[..., None])[..., 0], -1, 0)
        return a, np.zeros_like(a), s
    jT, djT, ddjT = j_esf(l, zT), dj_esf(l, zT), ddj_esf(l, zT)
    PB = -2 * mu * L * (zT * djT - jT)
    zero = np.zeros_like(x, dtype=complex)
    M = np.stack([np.stack([zL * djL, L * jT, -x * dh], -1),
                  np.stack([PA / rc, PB / rc, -x ** 2 * h], -1),
                  np.stack([2 * (zL * djL - jL), zT ** 2 * ddjT + (L - 2) * jT, zero], -1)], -2)
    rhs = np.stack([x * dj_esf(l, x), x ** 2 * j_esf(l, x), zero], -1).astype(complex)
    a, b, s = np.moveaxis(np.linalg.solve(M, rhs[..., None])[..., 0], -1, 0)
    return a, b, s


x_teste = np.array([0.05, 0.37, 0.92])
for l in (0, 1, 4):
    w = j_esf(l, x_teste) * dh_esf(l, x_teste) - dj_esf(l, x_teste) * h_esf(l, x_teste)
    print(f"l = {l}: max |j h' - j' h - i/x^2| = {np.max(np.abs(w - 1j / x_teste ** 2)):.2e}")

# limite fluido de a_l: c_T/c_L = 1e-7 (bolha de ar em água)
cL_ar, rho_ar = 343.0, 1.225
for l in (0, 1, 2):
    a, _, _ = coeficientes(x_teste, l, cL_ar, 1e-7 * cL_ar, rho_ar)
    zL = x_teste * CM / cL_ar
    a_fluido = 1j * RHOM / (x_teste ** 2 * rho_ar) / (j_esf(l, zL) * dh_esf(l, x_teste)
                                                     - (RHOM * CM) / (rho_ar * cL_ar) * dj_esf(l, zL) * h_esf(l, x_teste))
    print(f"l = {l}: max |a_l - a_l(fluido)|/|a_l(fluido)| = {np.max(np.abs(a - a_fluido) / np.abs(a_fluido)):.2e}")

## 3. Energia interna

A energia média armazenada é a soma da energia cinética e da elástica, $\langle W\rangle=\langle T\rangle+\langle U\rangle$. Pelo teorema
de Clapeyron harmônico, $\langle U\rangle=\langle T\rangle+\tfrac14\mathrm{Re}\{I_S\}$, em que $I_S$ é a integral de superfície de
$\mathbf t\cdot\mathbf u^*$, e as integrais radiais têm forma fechada (relações de Lommel):

$$
\begin{aligned}
\frac{2f_0\langle W\rangle}{Z_S|A|^2}={}&2\sum_{\ell=0}^{\infty}(2\ell+1)\Big\{|a_\ell|^2|z_L|\,G_\ell(z_L)
+|b_\ell|^2\ell(\ell+1)|z_L|\Big[|j_\ell(z_T)|^2+G_\ell(z_T)\Big]
+2\ell(\ell+1)\,\mathrm{Re}\big[a_\ell b_\ell^*|z_L|\,j_\ell(z_L)j_\ell^*(z_T)\big]\Big\}\\
&-\mathrm{Re}\Bigg\{\frac{1}{\rho_S|c_L|c_L^*}\sum_{\ell=0}^{\infty}(2\ell+1)\Big[a_\ell^*j_\ell'^*(z_L)+b_\ell^*\frac{\ell(\ell+1)}{z_L^*}j_\ell^*(z_T)\Big]
\Big[a_\ell z_L^2\big(\lambda j_\ell(z_L)-2\mu j_\ell''(z_L)\big)-2\mu\,\ell(\ell+1)\,b_\ell\big(z_Tj_\ell'(z_T)-j_\ell(z_T)\big)\Big]\Bigg\},
\end{aligned}
$$

com $Z_S=\rho_S|c_L|$ e

$$
G_\ell(z)=\frac{|z|^2}{z^2-(z^*)^2}\Big[z\,j_\ell(z)\,j_\ell'(z^*)-z^*j_\ell'(z)\,j_\ell(z^*)\Big].
$$

No limite sem perda ($z$ real) a expressão degenera e usa-se a relação de Lommel para argumento real,
$G_\ell(x)=x\,j_\ell j_\ell'+\tfrac{x^2}{2}\big[j_\ell^2-j_{\ell-1}j_{\ell+1}\big]$, isto é,
$x\,G_\ell(x)=\int_0^x\big[t^2j_\ell'^2(t)+\ell(\ell+1)j_\ell^2(t)\big]dt$. O gerador grava a energia com a impedância do meio,
$W_T=(Z_S/Z_M)\,2f_0\langle W\rangle/(Z_S|A|^2)$, que só muda a escala absoluta. A soma em $\ell$ é truncada em $\ell_{\max}=10$, como
no banco de dados. A célula seguinte confere a forma real de $G_\ell$ contra a integração numérica e a continuidade entre os dois ramos.

In [ ]:
def G_lommel(l, z, tol=1e-6):
    # ramo complexo e ramo real (|Im z| < 1e-6 min(1, |z|)) da integral de Lommel
    z = np.asarray(z, dtype=complex)
    real = np.abs(z.imag) < tol * np.minimum(1.0, np.abs(z))
    zc = np.where(real, z + 1j, z)              # o ramo complexo não é usado onde z é real
    j, dj = j_esf(l, zc), dj_esf(l, zc)
    Gc = np.abs(zc) ** 2 / (zc ** 2 - np.conj(zc) ** 2) * (zc * j * np.conj(dj) - np.conj(zc) * dj * np.conj(j))
    xr = z.real
    jr, djr = spherical_jn(l, xr), spherical_jn(l, xr, derivative=True)
    jm = spherical_jn(l - 1, xr) if l > 0 else np.cos(xr) / xr
    Gr = xr * jr * djr + 0.5 * xr ** 2 * (jr ** 2 - jm * spherical_jn(l + 1, xr))
    return np.where(real, Gr, Gc)


def energia_modal_esfera(xM, l, cL, cT, rhoS, cM=CM, rhoM=RHOM):
    # termo l de 2 f0 <W>/(Z_S |A|^2)
    x = np.asarray(xM, dtype=float)
    cL, cT = np.broadcast_to(cL, x.shape).astype(complex), np.broadcast_to(cT, x.shape).astype(complex)
    a, b, _ = coeficientes(x, l, cL, cT, rhoS, cM, rhoM)
    zL, zT = x * cM / cL, x * cM / cT
    mu = rhoS * cT ** 2
    lam = rhoS * cL ** 2 - 2 * mu
    L, f = l * (l + 1), 2 * l + 1
    jL, djL, ddjL = j_esf(l, zL), dj_esf(l, zL), ddj_esf(l, zL)
    xL = np.abs(zL)
    PA = zL ** 2 * (lam * jL - 2 * mu * ddjL)
    dois_T = 2 * f * np.abs(a) ** 2 * xL * G_lommel(l, zL)
    sup = np.conj(a) * np.conj(djL) * a * PA
    if l > 0:
        jT, djT = j_esf(l, zT), dj_esf(l, zT)
        PB = -2 * mu * L * (zT * djT - jT)
        dois_T = dois_T + 2 * f * (np.abs(b) ** 2 * L * xL * (np.abs(jT) ** 2 + G_lommel(l, zT))
                                   + 2 * L * np.real(a * np.conj(b) * xL * jL * np.conj(jT)))
        sup = (np.conj(a) * np.conj(djL) + np.conj(b) * L / np.conj(zL) * np.conj(jT)) * (a * PA + b * PB)
    return np.real(dois_T) - np.real(f * sup / (rhoS * np.abs(cL) * np.conj(cL)))


def energia_vel(xM, cL, cT, rhoS, cM=CM, rhoM=RHOM, lmax=L_MAX, normalizacao="ZM", por_l=False):
    # W_T = soma dos W_l para velocidades dadas; "ZM": 2 f0 W/(Z_M |A|^2), como no gerador; "ZS": 2 f0 W/(Z_S |A|^2), como no texto
    x = np.asarray(xM, dtype=float)
    cL, cT = np.broadcast_to(cL, x.shape).astype(complex), np.broadcast_to(cT, x.shape).astype(complex)
    with np.errstate(all="ignore"):
        Wl = np.array([energia_modal_esfera(x, l, cL, cT, rhoS, cM, rhoM) for l in range(lmax + 1)])
    if normalizacao == "ZM":
        Wl = Wl * rhoS * np.abs(cL) / (rhoM * cM)
    WT = Wl.sum(axis=0)
    return (WT, Wl) if por_l else WT


def energia_esfera(xM, modelo, cL0, cT0, rhoS, tau=0.0, lmax=L_MAX, normalizacao="ZM", por_l=False, cM=CM, rhoM=RHOM):
    cL, cT = velocidades(xM, modelo, cL0, cT0, tau)
    return energia_vel(xM, cL, cT, rhoS, cM, rhoM, lmax, normalizacao, por_l)


print(" l    x      x G_l(x)               integral numérica      continuidade |G(x+i1e-9) - G(x)|/|G(x)|")
for l in (0, 1, 3, 6):
    for x in (0.5, 4.0, 12.0):
        num = quad(lambda t: t ** 2 * dj_esf(l, t) ** 2 + l * (l + 1) * j_esf(l, t) ** 2, 0.0, x, limit=200)[0]
        G = float(np.real(G_lommel(l, x)))
        Gi = complex(G_lommel(l, x + 1e-9j, tol=0.0))
        print(f"{l:2d} {x:5.1f} {x * G:22.12e} {num:22.12e} {abs(Gi - G) / abs(G):12.2e}")

### Verificação pela integração no volume

A Eq. (5.58) também é conferida por uma rota independente (Fig. 23 da tese): a energia cinética e as parcelas elásticas
$\lambda|\nabla\cdot\mathbf u|^2$ e $2\mu\,\varepsilon_{ij}\varepsilon_{ij}^*$ ($rr$, $r\theta$ e $\theta\phi$) são escritas em termos de
integrais radiais de produtos de funções de Bessel, calculadas por quadratura de Gauss–Legendre ($200$ nós), sem as relações de
Lommel nem a redução à superfície. Os materiais são os da tese, com velocidades complexas de fase constante: tungstênio
($c_L=5200$ m/s, $c_T=2900$ m/s, sem perda), poliuretano ($1800$ e $800$ m/s, $-75^\circ$) e PMMA ($2700$ e $1330$ m/s, $-5^\circ$),
em água do mar ($c_M=1522$ m/s, $\rho_M=1024$ kg/m$^3$).

In [ ]:
from numpy.polynomial.legendre import leggauss
T_GL, P_GL = leggauss(200)


def integral_radial(f, xL):
    # integral de 0 a xL por Gauss-Legendre
    xi = 0.5 * xL * (T_GL + 1.0)
    return 0.5 * xL * np.sum(P_GL * f(xi))


def energia_volume(xM, cL, cT, rhoS, cM, rhoM, lmax=L_MAX):
    # 2 f0 <W>/(Z_S |A|^2) pelas parcelas integradas no volume: cinética + lambda + mu_rr + mu_rtheta + mu_thetaphi
    zL = cM / cL * xM
    xL = abs(zL)
    f, fc = zL / xL, np.conj(zL / xL)
    eta = cL / cT
    etc = np.conj(eta)
    mu = rhoS * cT ** 2
    lam = rhoS * cL ** 2 - 2 * mu
    K = Lam = Mrr = Mrt = Mtp = 0j
    for l in range(lmax + 1):
        a, b, _ = coeficientes(np.array([xM]), l, cL, cT, rhoS, cM, rhoM)
        a, b = a[0], b[0]
        L, g, K2 = l * (l + 1), 2 * l + 1, l * (l + 1) - 2
        I = lambda fun: integral_radial(fun, xL)
        j = lambda z: j_esf(l, z)
        dj = lambda z: dj_esf(l, z)
        ddj = lambda z: ddj_esf(l, z)
        jT = lambda x: j(eta * f * x)
        jTc = lambda x: j(etc * fc * x)
        dT = lambda x: jT(x) + eta * f * x * dj(eta * f * x)          # j(z_T r) + z_T r j'(z_T r)
        dTc = lambda x: jTc(x) + etc * fc * x * dj(etc * fc * x)
        # cinética
        I1, I4 = I(lambda x: x ** 2 * dj(f * x) * dj(fc * x)), I(lambda x: j(f * x) * j(fc * x))
        I2, I3 = I(lambda x: jT(x) * jTc(x)), I(lambda x: x * dj(f * x) * jTc(x))
        I5, I6 = I(lambda x: dT(x) * dTc(x)), I(lambda x: j(f * x) * dTc(x))
        K += g * (abs(a) ** 2 * I1 + L ** 2 * abs(b) ** 2 * I2 + 2 * L * np.real(a * np.conj(b) * f * I3)
                  + L * (abs(a) ** 2 * I4 + abs(b) ** 2 * I5 + 2 * np.real(a * np.conj(b) * I6)))
        # lambda |div u|^2
        Lam += g * abs(a) ** 2 * I(lambda x: x ** 2 * j(f * x) * j(fc * x))
        # mu, componente rr
        Mrr_l = abs(a) ** 2 * I(lambda x: x ** 2 * ddj(f * x) * ddj(fc * x))
        if l > 0:
            I2 = I(lambda x: dj(eta * f * x) * dj(etc * fc * x))
            I3 = I(lambda x: jT(x) * dj(etc * fc * x) / x)
            I3h = I(lambda x: dj(eta * f * x) * jTc(x) / x)
            I4 = I(lambda x: jT(x) * jTc(x) / x ** 2)
            I5 = I(lambda x: x * ddj(f * x) * dj(etc * fc * x))
            I6 = I(lambda x: ddj(f * x) * jTc(x))
            Mrr_l += abs(b) ** 2 * L ** 2 * (abs(eta) ** 2 * I2 - eta * f * I3h - etc * fc * I3 + I4)
            Mrr_l += 2 * L * np.real(a * np.conj(b) * (etc * f * I5 - f ** 2 * I6))
            # mu, componente r theta
            J1, J4 = I(lambda x: dj(f * x) * dj(fc * x)), I(lambda x: j(f * x) * j(fc * x) / x ** 2)
            J2, J3 = I(lambda x: j(f * x) * dj(fc * x) / x), I(lambda x: dj(f * x) * j(fc * x) / x)
            J5 = I(lambda x: x ** 2 * ddj(eta * f * x) * ddj(etc * fc * x))
            J6, J7 = I(lambda x: ddj(eta * f * x) * jTc(x)), I(lambda x: jT(x) * ddj(etc * fc * x))
            J8 = I(lambda x: jT(x) * jTc(x) / x ** 2)
            J9, J10 = I(lambda x: x * dj(f * x) * ddj(etc * fc * x)), I(lambda x: dj(f * x) * jTc(x) / x)
            J11, J12 = I(lambda x: j(f * x) * ddj(etc * fc * x)), I(lambda x: j(f * x) * jTc(x) / x ** 2)
            ta = 4 * J1 - 4 * fc * J2 - 4 * f * J3 + 4 * J4
            tb = abs(eta) ** 4 * J5 + K2 * eta ** 2 * f ** 2 * J6 + K2 * etc ** 2 * fc ** 2 * J7 + K2 ** 2 * J8
            tc = 2 * np.real(a * np.conj(b) * (2 * etc ** 2 * fc * J9 + 2 * K2 * f * J10 - 2 * etc ** 2 * fc ** 2 * J11 - 2 * K2 * J12))
            Mrt += L * g * (abs(a) ** 2 * ta + abs(b) ** 2 * tb + tc)
        Mrr += g * Mrr_l
        # mu, componentes theta theta e phi phi
        A = lambda x: a * dj(f * x) + b * (L / (f * x)) * jT(x)
        B = lambda x: a * j(f * x) / (f * x) + b * (jT(x) / (f * x) + eta * dj(eta * f * x))
        IAA = I(lambda x: A(x) * np.conj(A(x)))
        if l == 0:
            Mtp += g * 4 * IAA
        else:
            IAB, IBB = I(lambda x: A(x) * np.conj(B(x))), I(lambda x: B(x) * np.conj(B(x)))
            Mtp += g * (4 * IAA - 2 * L * (IAB + np.conj(IAB)) + L ** 2 * IBB)
            if l >= 2:
                Mtp += g * np.prod(np.arange(l - 1, l + 3)) * IBB              # (l+2)!/(l-2)!
    den = rhoS * abs(cL) ** 2
    return float(np.real(K + lam / den * Lam + 2 * mu / den * Mrr + mu / den * Mrt + mu / den * Mtp))


C_23, RHO_23 = 1522.0, 1024.0                         # água do mar
MATERIAIS_23 = {"Tungstênio": (5200.0, 2900.0, 19300.0, 0.0), "Poliuretano": (1800.0, 800.0, 1200.0, -75.0),
                "PMMA (acrílico)": (2700.0, 1330.0, 1180.0, -5.0)}
x_ana, x_num = np.linspace(0.01, 10.0, 2400), np.linspace(0.30, 10.0, 12)
W_ana, W_num = {}, {}
for nome, (cl, ct, rho, fase) in MATERIAIS_23.items():
    e = np.exp(1j * np.deg2rad(fase))
    W_ana[nome] = energia_vel(x_ana, cl * e, ct * e, rho, C_23, RHO_23, normalizacao="ZS")
    W_num[nome] = np.array([energia_volume(x, cl * e, ct * e, rho, C_23, RHO_23) for x in x_num])
    W_ref = energia_vel(x_num, cl * e, ct * e, rho, C_23, RHO_23, normalizacao="ZS")
    print(f"{nome:16s}: diferença relativa máxima entre a Eq. (5.58) e a integração no volume = "
          f"{np.max(np.abs(W_num[nome] - W_ref) / np.abs(W_ref)):.1e}")

fig, ax = plt.subplots(figsize=(5, 3.5))
plotar_curvas(ax, x_ana, list(W_ana.values()), list(W_ana))
for W in W_num.values():
    ax.plot(x_num, W, linestyle="none", marker="o", markersize=3.2, markerfacecolor="white", markeredgecolor="black",
            markeredgewidth=0.7, zorder=10)
eixo_energia(ax, (0.0, 10.0), [0, 3, 6, 9], (1e-5, 1e5), [1e-4, 1e-2, 1e0, 1e2, 1e4], casas=0)
h, r = ax.get_legend_handles_labels()
h.append(Line2D([0], [0], linestyle="none", marker="o", markersize=3.5, markerfacecolor="white", markeredgecolor="black",
                markeredgewidth=0.7))
r.append("Integração no volume")
adicionar_legenda(ax, handles=h, labels=r)
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.13, top=0.98)
salvar_figura(fig, "energia_total_3_materiais")
plt.show()
plt.close(fig)

## 4. Comparação com os casos-limite

**Esfera fluida.** Para $c_T\to0$ ($\mu\to0$), $z_T\to\infty$, $R_\ell\to0$ e $b_\ell\to0$; o coeficiente longitudinal reduz-se à
Eq. (5.12) e a energia da esfera sólida deve tender à da esfera fluida de mesma densidade e velocidade $c_S=c_L$
(capítulo da esfera fluida, com $m=c_M/c_L$ e $\rho=\rho_S/\rho_M$):

$$
\frac{2f_0W_T}{Z_S|A|^2}=\sum_{\ell=0}^{\infty}|b_\ell^{\mathrm{fl}}|^2(2\ell+1)\Big[2F_\ell(x_S)+x_S^2\,j_\ell(x_S)\,j_\ell'(x_S)\Big],\qquad
F_\ell(x_S)=\frac{x_S^3}{2}\Big[j_\ell^2(x_S)-j_{\ell-1}(x_S)\,j_{\ell+1}(x_S)\Big].
$$

**Esfera elástica.** No Kelvin–Voigt, $c_j\to c_{j,0}$ quando $\tilde\tau_V\to0$; no Maxwell, $c_j\to c_{j,\infty}$ quando
$\tilde\tau_M\to\infty$ (Eq. 5.101). No limite fluido, a diferença relativa deve cair como $(c_T/c_L)^2$. Nos limites viscoelásticos
a perda entra como parte imaginária dos módulos, e a diferença cai entre a primeira e a segunda potência de $\tilde\tau_V$ (ou de
$1/\tilde\tau_M$), conforme o peso das ressonâncias estreitas na curva. A célula abaixo mede a diferença relativa mediana de $W_T$ em
$10^{-4}\le x_M\le1$ (2000 pontos) e o percentil 95, para uma bolha de ar e uma esfera macia ($c_L=1000$ m/s, $\rho_S=1070$ kg/m$^3$
e, no caso viscoelástico, $c_T=50$ m/s, a esfera macia da Fig. 25) e para o silicone poroso da Fig. 30. Os picos transversais muito
estreitos dominam o máximo, mas também convergem quando o parâmetro diminui.

In [ ]:
def energia_fluida(xM, m, rho_ratio, lmax=L_MAX):
    # esfera fluida: 2 f0 W_T/(Z_S |A|^2), x_S = m x_M, b_l = i x_M^-2 (rho_M/rho_S)/D_l
    x = np.asarray(xM, dtype=float)
    xS = m * x
    W = 0.0
    for l in range(lmax + 1):
        jm = spherical_jn(l - 1, xS) if l > 0 else np.cos(xS) / xS
        D = dh_esf(l, x) * j_esf(l, xS) - (m / rho_ratio) * dj_esf(l, xS) * h_esf(l, x)
        b = 1j / (rho_ratio * x ** 2 * D)
        F = 0.5 * xS ** 3 * (j_esf(l, xS) ** 2 - jm * j_esf(l + 1, xS))
        W = W + (2 * l + 1) * np.abs(b) ** 2 * (2 * F + xS ** 2 * j_esf(l, xS) * dj_esf(l, xS))
    return W


def diferenca(W, W_ref):
    r = np.abs(W - W_ref) / np.abs(W_ref)
    r = r[np.isfinite(r)]
    return np.median(r), np.quantile(r, 0.95)


X_LIM = np.unique(np.concatenate([np.geomspace(1e-4, 0.1, 600), np.linspace(0.1, 1.0, 1400)]))
EPS_T = np.array([1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])            # c_T/c_L
TAU_V = np.array([1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8])            # tau_V (Kelvin-Voigt)
TAU_M = 1.0 / TAU_V                                               # tau_M (Maxwell)

FLUIDOS = {"Bolha de ar": dict(cL=343.0, rhoS=1.225), "Esfera macia": dict(cL=1000.0, rhoS=1070.0)}
SOLIDOS = {"Silicone poroso": dict(cL0=120.0, cT0=40.0, rhoS=760.0), "Esfera macia": dict(cL0=1000.0, cT0=50.0, rhoS=1070.0)}

LIM = {}
for nome, p in FLUIDOS.items():
    W_ref = energia_fluida(X_LIM, CM / p["cL"], p["rhoS"] / RHOM)
    LIM[("fluido", nome)] = np.array([diferenca(energia_esfera(X_LIM, "elastico", p["cL"], e * p["cL"], p["rhoS"],
                                                               normalizacao="ZS"), W_ref) for e in EPS_T])
for nome, p in SOLIDOS.items():
    W_ref = energia_esfera(X_LIM, "elastico", p["cL0"], p["cT0"], p["rhoS"])
    LIM[("voigt", nome)] = np.array([diferenca(energia_esfera(X_LIM, "voigt", p["cL0"], p["cT0"], p["rhoS"], t), W_ref) for t in TAU_V])
    LIM[("maxwell", nome)] = np.array([diferenca(energia_esfera(X_LIM, "maxwell", p["cL0"], p["cT0"], p["rhoS"], t), W_ref) for t in TAU_M])

for (tipo, nome), v in LIM.items():
    par, valores_par = {"fluido": ("c_T/c_L", EPS_T), "voigt": ("tau_V", TAU_V), "maxwell": ("1/tau_M", 1 / TAU_M)}[tipo]
    print(f"{tipo:8s} {nome:16s} {par:8s}: " + "  ".join(f"{p:.0e}: {m:.1e} (p95 {q:.1e})" for p, (m, q) in zip(valores_par, v)))
    inc = np.polyfit(np.log10(valores_par[2:5]), np.log10(v[2:5, 0]), 1)[0]
    print(f"{'':35s}inclinação em log-log (3o ao 5o valor): {inc:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
ax = axes[0]
for i, nome in enumerate(FLUIDOS):
    ax.plot(EPS_T, LIM[("fluido", nome)][:, 0], color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i],
            marker="o", markersize=4, label=nome)
ax.plot(EPS_T, LIM[("fluido", "Bolha de ar")][-1, 0] * (EPS_T / EPS_T[-1]) ** 2, color="0.45", linestyle=":", linewidth=1.6,
        label=r"$\propto(c_T/c_L)^2$")
ax.set_xscale("log"); ax.set_yscale("log")
estilizar(ax, r"$c_T/c_L$", r"Diferença relativa mediana de $W_T$")
adicionar_caixa(ax, r"Esfera fluida, $c_T\to0$", x=0.035, ha="left")
adicionar_legenda(ax)
ax = axes[1]
for k, (tipo, rot) in enumerate((("voigt", "Kelvin-Voigt"), ("maxwell", "Maxwell"))):
    for i, nome in enumerate(SOLIDOS):
        ax.plot(TAU_V, LIM[(tipo, nome)][:, 0], color=COLORS[k], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i],
                marker="o" if i == 0 else "s", markersize=4, label=f"{rot}, {nome.lower()}")
ax.plot(TAU_V, LIM[("voigt", "Silicone poroso")][-1, 0] * TAU_V / TAU_V[-1], color="0.45", linestyle=":", linewidth=1.6,
        label=r"$\propto\tilde\tau_V,\ 1/\tilde\tau_M$")
ax.set_xscale("log"); ax.set_yscale("log")
estilizar(ax, r"$\tilde\tau_V$ ou $1/\tilde\tau_M$")
adicionar_caixa(ax, r"Esfera elástica", x=0.035, ha="left")
adicionar_legenda(ax, fontsize=8)
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.16, top=0.98, wspace=0.08)
salvar_figura(fig, "grafico_limites")
plt.show()
plt.close(fig)

linhas = list(FLUIDOS)
dados = [[f"${sci(m)}$" for m, _ in LIM[("fluido", n)]] for n in linhas]
valores = pd.DataFrame([LIM[("fluido", n)][:, 0] for n in linhas], index=linhas, columns=[f"cT/cL={e:g}" for e in EPS_T])
salvar_tabela_texto("tabela_limite_fluido", linhas, [f"${sci(e, 0)}$" for e in EPS_T], dados, valores, figsize=(10.5, 1.6),
                    first_col_width=0.19, header_label=r"$c_T/c_L$")
linhas, dados, val = [], [], []
for tipo, rot in (("voigt", "Kelvin-Voigt"), ("maxwell", "Maxwell")):
    for nome in SOLIDOS:
        linhas.append(f"{rot}, {nome.lower()}")
        dados.append([f"${sci(m)}$" for m, _ in LIM[(tipo, nome)]])
        val.append(LIM[(tipo, nome)][:, 0])
valores = pd.DataFrame(val, index=linhas, columns=[f"tauV_ou_1/tauM={t:g}" for t in TAU_V])
salvar_tabela_texto("tabela_limite_elastico", linhas, [f"${sci(t, 0)}$" for t in TAU_V], dados, valores, figsize=(11.5, 2.4),
                    first_col_width=0.27, header_label=r"$\tilde\tau_V$ ou $1/\tilde\tau_M$")

Os mesmos limites aparecem nas curvas de energia da Fig. 25 da tese: a bolha de ar com $c_T/c_L=10^{-5}$ contra a esfera fluida,
e a esfera macia com $c_T$ finito contra a esfera fluida de mesma densidade e $c_S=c_L$.

In [ ]:
C_25, RHO_25 = 1522.0, 1024.0                          # água do mar (Fig. 25)
x25 = np.unique(np.concatenate([np.geomspace(1e-5, 0.02, 4000, endpoint=False), np.linspace(0.02, 1.0, 8000)]))

# bolha de ar: c_T/c_L = 1e-5 no modelo elástico contra a esfera fluida
c_ar, rho_ar = 343.0, 1.225
W_ar_el = energia_vel(x25, c_ar, 1e-5 * c_ar, rho_ar, C_25, RHO_25, normalizacao="ZS")
W_ar_fl = energia_fluida(x25, C_25 / c_ar, rho_ar / RHO_25)
figura_curvas(x25, [W_ar_el, W_ar_fl], [r"Elástico, $c_T\to0$", "Fluido"], "bolha_limite_ct", (0.0, 1.0),
              [0.0, 0.3, 0.6, 0.9], (1e-5, 1e10), [1e-5, 1e0, 1e5, 1e10], casas=1,
              caixa=rf"$\rho_S/\rho_M={fmt_sig(rho_ar / RHO_25, 3)}$" + "\n" + rf"$c_M/c_S={dec(C_25 / c_ar, 2)}$")

# esfera macia: c_T finito contra a esfera fluida de mesma densidade e c_S = c_L
cL_mac, cT_mac, rho_mac = 1000.0, 50.0, 1100.0
W_mac_el = energia_vel(x25, cL_mac, cT_mac, rho_mac, C_25, RHO_25, normalizacao="ZS")
W_mac_fl = energia_fluida(x25, C_25 / cL_mac, rho_mac / RHO_25)
figura_curvas(x25, [W_mac_el, W_mac_fl], ["Elástico", "Fluido equivalente"], "solido_macio", (0.0, 0.15),
              [0.0, 0.05, 0.10, 0.15], (1e-7, 1e3), [1e-6, 1e-3, 1e0, 1e3], casas=2,
              caixa=rf"$\rho_S/\rho_M={dec(rho_mac / RHO_25, 2)}$" + "\n" + rf"$c_L/c_M={dec(cL_mac / C_25, 3)}$"
                    + "\n" + rf"$\eta={cL_mac / cT_mac:g}$")

## 5. Curvas de energia

Comparação entre a esfera elástica e as descrições de Kelvin–Voigt e de Maxwell, com as mesmas propriedades elásticas de referência
(microesferas de silicone poroso de Tallon et al., $c_L=120$ m/s, $c_T=40$ m/s e $\rho_S=760$ kg/m$^3$, em $c_M=1490$ m/s e
$\rho_M=1000$ kg/m$^3$), Fig. 30 da tese. A perda é a mesma nos dois modelos em $x_M=0{,}5$ ($\tan\delta_T=0{,}3$), o que
dá $\tilde\tau_V=0{,}6$ e $\tilde\tau_M=6{,}67$; a soma em $\ell$ vai até $\ell=12$, porque $|z_T|$ chega a $9$ em $x_M=0{,}25$.
O segundo gráfico repete o espectro com $\log_{10}(W/W_{\mathrm{el}})$ embaixo.

In [ ]:
C_30, RHO_30 = 1490.0, 1000.0                          # gel aquoso de Tallon et al.
cL_30, cT_30, rho_30 = 120.0, 40.0, 760.0              # silicone poroso
TAU_V30, TAU_M30 = 0.60, 1.0 / (0.30 * 0.50)           # tan d_T = 0,3 em x_M = 0,5 nos dois modelos
x30 = np.unique(np.concatenate([np.geomspace(1e-4, 0.05, 2200, endpoint=False), np.linspace(0.05, 0.25, 5200)]))
W30 = {m: energia_esfera(x30, m, cL_30, cT_30, rho_30, t, lmax=12, normalizacao="ZS", cM=C_30, rhoM=RHO_30)
       for m, t in (("elastico", 0.0), ("voigt", TAU_V30), ("maxwell", TAU_M30))}
ROT30 = ["Elástico", "Kelvin-Voigt", "Maxwell"]
print(f"c_L/c_M = {cL_30 / C_30:.4g} | rho_S/rho_M = {rho_30 / RHO_30:.3g} | eta = {cL_30 / cT_30:g} | "
      f"tau_V = {TAU_V30:g} | tau_M = {TAU_M30:.4g}")
CAIXA30 = (rf"$\rho_S/\rho_M={dec(rho_30 / RHO_30, 2)}$" + "\n" + rf"$c_L/c_M={dec(cL_30 / C_30, 3)}$" + "\n"
           + rf"$\eta={cL_30 / cT_30:g}$")
figura_curvas(x30, list(W30.values()), ROT30, "comparacao_elastico_voigt_maxwell", (0.0, 0.25), [0.0, 0.125, 0.25],
              (1e-5, 1e7), [1e-4, 1e-1, 1e3, 1e7], casas=3, caixa=CAIXA30)

# mesmo espectro com a razão em relação ao elástico
fig = plt.figure(figsize=(5, 3.5))
gs = fig.add_gridspec(2, 1, height_ratios=[3.4, 1.0], hspace=0.06)
ax, axr = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
plotar_curvas(ax, x30, list(W30.values()), ROT30)
eixo_energia(ax, (0.0, 0.25), [0.0, 0.125, 0.25], (1e-5, 1e7), [1e-4, 1e-1, 1e3, 1e7], casas=3)
ax.set_xlabel(None)
plt.setp(ax.get_xticklabels(), visible=False)
adicionar_legenda(ax)
with np.errstate(all="ignore"):
    for i, m in enumerate(("voigt", "maxwell"), start=1):
        axr.plot(x30, np.log10(W30[m] / W30["elastico"]), color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i])
axr.axhline(0.0, color=COLORS[0], linestyle=LINESTYLES[0], linewidth=1.0, zorder=0)
axr.set_xlim(0.0, 0.25)
axr.xaxis.set_major_locator(FixedLocator([0.0, 0.125, 0.25]))
axr.set_xticklabels([rotulo_x(v, 3) for v in (0.0, 0.125, 0.25)])
estilizar(axr, r"$x_M=k_Ma$", r"$\log_{10}(W/W_{\mathrm{el}})$")
axr.yaxis.label.set_size(10)
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.13, top=0.98)
salvar_figura(fig, "comparacao_elastico_voigt_maxwell_razao")
plt.show()
plt.close(fig)

## 6. Banco de dados e contagem das ressonâncias

O gerador (`gerador_viscoelastico.py`, versão 2) calcula $W_T$ em $10^{-4}\le x_M\le1$ com $0\le\ell\le10$ para curvas sorteadas
num hipercubo, separadamente para cada modelo: $\alpha=\rho_M/\rho_S$ e $\beta=c_M/c_L$ log-uniformes, $c_T$ uniforme entre
$10^{-3}$ m/s e $0{,}999\,(\sqrt3/2)\,c_L$ (isto é, $\eta\ge2/\sqrt3$) e, nos modelos viscoelásticos, $\tilde\tau$ log-uniforme.
As faixas exatas e as opções de cada rodada são lidas de `parametros_viscoelastico_lmax10_*.json` e impressas abaixo.
Nos modelos viscoelásticos, $c_L$, $c_T$ e $\eta$ das tabelas são as velocidades de referência: $c_{j,0}$ no Kelvin–Voigt e
$c_{j,\infty}$ no Maxwell.

Uma ressonância é contada assim: nas curvas em que a condição de recorrência não prevê nenhum polo com $Q\ge1$ até $x_M=1{,}1$,
cada máximo de $W_T$ na malha base de $5\,000$ pontos com proeminência de pelo menos $0{,}1$ em $\log_{10}W_T$; nas demais (curvas
de risco), os zeros de $D_\ell$ no plano complexo, $x_M=x_k-i\gamma_k$, com $Q=x_k/(2\gamma_k)\ge2$ que são visíveis no modelo
analítico da curva ou carregam pelo menos $3\times10^{-4}$ da energia da janela $\Delta_W$. Se a busca de polos não fecha em todos os
$\ell$, a contagem fica com os máximos da malha refinada. $N_{\mathrm{res}}$ é essa contagem final (`N_res`), e cada ressonância
traz a sua posição $x_M$ e a ordem $\ell$ do polo associado (`hipercubo_*.npz`).

**Máximos espúrios em sólidos muito rígidos.** Nas curvas normais de sólidos com $c_L/c_M>100$, a malha base pode registrar
máximos em $x_M<10^{-3}$ com $W_T$ de $10^{90}$ ou mais. São estouros numéricos do gerador: a energia recalculada aqui nesses pontos
é suave e da ordem de $10^{-13}$, e a equação característica (Seção 9) não tem nenhuma raiz em $x_M\le1$ nesses sólidos. A célula
abaixo lista essas curvas. Com `EXCLUIR_SUSPEITAS = True` elas saem de todas as seções seguintes; o padrão é `False`, para que o banco
não seja alterado sem uma decisão explícita.

**Consistência com o gerador.** No fim da célula, $W_T$ calculado pelas funções das Seções 2 e 3 é comparado com a altura que o
gerador gravou em picos bem resolvidos da malha ($Q\le10^4$ e $|z_T|\le50$).

**Busca dos arquivos.** A pasta `PASTA` é percorrida inteira, inclusive subpastas cujo nome só começa igual ao da pasta de saída
(por exemplo, `resultados_mix`); só a própria pasta de saída fica de fora. Se algum arquivo não for encontrado, a execução para com
uma mensagem que diz onde se procurou e quais subpastas existem.

In [ ]:
MODELOS = ("elastico", "voigt", "maxwell")
NOME_MODELO = {"elastico": "Elástico", "voigt": "Kelvin-Voigt", "maxwell": "Maxwell"}
TEX_MODELO = {"elastico": "Elástico", "voigt": "Kelvin-Voigt", "maxwell": "Maxwell"}
PADROES = {"resumo": "resumo_viscoelastico_lmax10_{m}.csv", "hiper": "hipercubo_{m}.npz",
           "param": "parametros_viscoelastico_lmax10_{m}.json"}


def procurar(nome_arquivo, pasta_modelo):
    achados = []
    saida = os.path.abspath(SAIDA)
    for raiz, dirs, arqs in os.walk(PASTA):
        # só a pasta de saída (caminho exato) fica de fora; "resultados_mix", "resultados_antigos" etc. são percorridas
        dirs[:] = sorted(d for d in dirs if os.path.abspath(os.path.join(raiz, d)) != saida)
        if nome_arquivo in arqs:
            achados.append(os.path.join(raiz, nome_arquivo))
    # prefere a pasta com o nome do modelo e, depois, o caminho mais curto
    achados.sort(key=lambda p: (os.path.basename(os.path.dirname(p)) != pasta_modelo, len(p), p))
    return achados


ARQ, faltando = {}, []
for m in MODELOS:
    ARQ[m] = {}
    for tipo, padrao in PADROES.items():
        achados = procurar(padrao.format(m=m), f"viscoelastico_lmax10_{m}")
        ARQ[m][tipo] = achados[0] if achados else None
        if not achados:
            faltando.append(padrao.format(m=m))
        elif len(achados) > 1:
            print(f"aviso: {len(achados)} cópias de {padrao.format(m=m)}; usando {achados[0]}")

if faltando:
    if os.path.isdir(PASTA):
        sub = sorted(os.path.relpath(os.path.join(r, d), PASTA) for r, ds, _ in os.walk(PASTA) for d in ds
                     if os.path.relpath(os.path.join(r, d), PASTA).count(os.sep) < 2)
        lista = "\n".join("    " + s for s in sub) if sub else "    (nenhuma subpasta)"
        msg = f"Procurei em {PASTA} (e em todas as subpastas, exceto {SAIDA}).\nSubpastas existentes:\n{lista}"
    else:
        msg = f"A pasta {PASTA} não existe. Defina VISCO_DADOS (fora do Colab) ou confira o nome da pasta no Drive."
    raise FileNotFoundError("Não encontrei: " + ", ".join(faltando) + "\n" + msg)

for m in MODELOS:
    print(f"{NOME_MODELO[m]}:")
    for tipo, p in ARQ[m].items():
        print(f"   {tipo:7s} {os.path.relpath(p, PASTA)}")

PARAM_GERADOR = {m: json.load(open(ARQ[m]["param"], encoding="utf-8")) for m in MODELOS}
for m in MODELOS:
    p = PARAM_GERADOR[m]
    print(f"\n{NOME_MODELO[m]}: versão {p['versao']}, semente {p['seed']}, {p['curvas_ok']} de {p['curvas_total']} curvas ok")
    print("   faixas:", p["constantes"]["FAIXAS"][m], "| x_M em", [p["constantes"]["X_MIN"], p["constantes"]["X_MAX"]],
          "| l_max =", p["constantes"]["LMAX"], "| c_M =", p["constantes"]["CM_PADRAO"], "| rho_M =", p["constantes"]["RHOM_PADRAO"])


def ler_resumo(m):
    r = pd.read_csv(ARQ[m]["resumo"])
    r = r.loc[r["status"] == "ok"].sort_values("_indice").reset_index(drop=True)
    r["_indice"] = r["_indice"].astype(int)
    r["c_razao"] = 1.0 / r["beta"]                                # c_L/c_M
    r["rho_ratio"] = 1.0 / r["alpha"]                             # rho_S/rho_M
    r["u"] = np.log(r["beta"])                                    # ln(c_M/c_L)
    r["v"] = np.log(r["rho_ratio"])                               # ln(rho_S/rho_M)
    r["w"] = np.log(r["eta"])                                     # ln eta
    r["N"] = r["N_res"].astype(int)
    r["ressonante"] = (r["N"] > 0).astype(int)
    return r


def ler_picos(m):
    with np.load(ARQ[m]["hiper"], allow_pickle=True) as d:
        df = pd.DataFrame({"curva": d["_indice"].astype(int), "x": d["xM_pico"].astype(float),
                           "l": pd.to_numeric(pd.Series(d["l"]), errors="coerce").to_numpy(float),
                           "altura": d["altura"].astype(float), "Q": d["Q"].astype(float), "origem": d["origem"].astype(str)})
    df = df.loc[np.isfinite(df["x"]) & (df["x"] >= 1e-4) & (df["x"] <= 1.0)]
    return df.sort_values(["curva", "x"]).reset_index(drop=True)


RESUMO = {m: ler_resumo(m) for m in MODELOS}
PICOS = {m: ler_picos(m) for m in MODELOS}

# diagnóstico: máximos espúrios da malha base em sólidos muito rígidos (ver o texto acima)
EXCLUIR_SUSPEITAS = True        # True: tira essas curvas de todas as seções seguintes
LIMIAR_ESTOURO = 1e30            # W_T acima disso num máximo da malha base é estouro numérico


def curvas_suspeitas(m):
    r, pk = RESUMO[m], PICOS[m]
    base = pk.loc[pk["origem"] == "malha_base"]
    estouro = set(base.loc[base["altura"] > LIMIAR_ESTOURO, "curva"])
    rigido = set(r.loc[(r["modo_contagem"] == "malha_base") & (r["N"] > 0) & (r["x_primeira_res"] < 1e-3)
                       & (r["c_razao"] > 100), "_indice"])
    return sorted(estouro | rigido)


SUSPEITAS = {m: curvas_suspeitas(m) for m in MODELOS}
for m in MODELOS:
    n_res = int(RESUMO[m]["ressonante"].sum())
    print(f"{NOME_MODELO[m]}: {len(SUSPEITAS[m])} curvas com máximos espúrios em x_M < 1e-3 "
          f"({100 * len(SUSPEITAS[m]) / max(n_res, 1):.1f} % das curvas ressonantes)"
          + (f", por exemplo {SUSPEITAS[m][:8]}" if SUSPEITAS[m] else ""))
if EXCLUIR_SUSPEITAS:
    for m in MODELOS:
        RESUMO[m] = RESUMO[m].loc[~RESUMO[m]["_indice"].isin(SUSPEITAS[m])].reset_index(drop=True)
        PICOS[m] = PICOS[m].loc[~PICOS[m]["curva"].isin(SUSPEITAS[m])].reset_index(drop=True)
    print("curvas suspeitas excluídas")
else:
    print("curvas suspeitas mantidas (EXCLUIR_SUSPEITAS = True)")

linhas_tab, val_tab = [], {}
for m in MODELOS:
    r, pk = RESUMO[m], PICOS[m]
    val_tab[m] = {"curvas": len(r), "ressonantes_%": 100 * r["ressonante"].mean(), "ressonancias": int(r["N"].sum()),
                  "maximo_por_curva": int(r["N"].max()), "contagem_polos_%": 100 * np.mean(r["modo_contagem"] == "polos_area"),
                  "com_l_%": 100 * np.isfinite(pk["l"]).mean() if len(pk) else np.nan,
                  "tempo_mediano_s": r["runtime_s"].median(), "tempo_medio_s": r["runtime_s"].mean()}
    print(f"\n{NOME_MODELO[m]}: {len(r)} curvas | {100 * r['ressonante'].mean():.1f} % com ressonância | "
          f"{int(r['N'].sum())} ressonâncias | máximo por curva {int(r['N'].max())} | tempo médio {r['runtime_s'].mean():.2f} s "
          f"(mediana {r['runtime_s'].median():.2f} s)")
    print("   modo de contagem:", r["modo_contagem"].value_counts().to_dict())
    print("   busca de polos:  ", r["busca_polos"].value_counts().to_dict())
    print(f"   picos com x_M em [1e-4, 1] no hipercubo: {len(pk)} ({val_tab[m]['com_l_%']:.1f} % com l do polo associado); "
          f"soma de N_res: {int(r['N'].sum())}")

VAL = pd.DataFrame(val_tab)
rot = {"curvas": "Curvas", "ressonantes_%": "Com ressonância (" + PCT + ")", "ressonancias": "Ressonâncias",
       "maximo_por_curva": "Máximo por curva", "contagem_polos_%": "Contagem pelos polos (" + PCT + ")",
       "tempo_mediano_s": "Tempo mediano (s)"}
fmt = {"curvas": lambda v: f"${milhar(v)}$", "ressonantes_%": lambda v: f"${dec(v, 1)}$", "ressonancias": lambda v: f"${milhar(v)}$",
       "maximo_por_curva": lambda v: f"${milhar(v)}$", "contagem_polos_%": lambda v: f"${dec(v, 1)}$",
       "tempo_mediano_s": lambda v: f"${dec(v, 2)}$"}
salvar_tabela_texto("tabela_banco_de_dados", [rot[k] for k in rot], [TEX_MODELO[m] for m in MODELOS],
                    [[fmt[k](VAL.at[k, m]) for m in MODELOS] for k in rot], VAL, figsize=(9.0, 3.4), first_col_width=0.40)

# consistência: W_T deste notebook no máximo de picos achados na malha (base ou refinada) contra a altura gravada pelo gerador.
# Só picos com Q <= 1e4 e |z_T| <= 50: em picos mais estreitos ou sólidos mais lentos, a solução direta do sistema da Seção 2
# perde precisão (o gerador usa D_l sem divisões e funções de Bessel escaladas) e a altura num ponto fixo da malha muda por
# ordens de grandeza com deslocamentos mínimos do polo.
for m in MODELOS:
    par = RESUMO[m].set_index("_indice")
    pk = PICOS[m].loc[PICOS[m]["origem"].str.startswith("malha") & ~PICOS[m]["curva"].isin(SUSPEITAS[m])
                      & PICOS[m]["curva"].isin(par.index) & (PICOS[m]["Q"] <= 1e4)]
    if len(pk):
        cT_pico = np.array([velocidades(np.array([x]), m, par.at[c, "cL0"], par.at[c, "cT0"], par.at[c, "tau_adim"])[1][0]
                            for c, x in zip(pk["curva"], pk["x"])])
        pk = pk.loc[np.abs(pk["x"].to_numpy() * CM / cT_pico) <= 50]
    pk = pk.sample(min(30, len(pk)), random_state=0) if len(pk) else pk
    dif = [abs(energia_esfera(np.array([x]), m, par.at[c, "cL0"], par.at[c, "cT0"], par.at[c, "rhoS"], par.at[c, "tau_adim"])[0] - h) / h
           for c, x, h in zip(pk["curva"], pk["x"], pk["altura"])]
    if dif:
        print(f"{NOME_MODELO[m]}: W_T deste notebook x altura do gerador em {len(dif)} picos da malha (fora das curvas suspeitas): "
              f"diferença relativa máxima {max(dif):.1e}")

### Curvas individuais do banco

Curvas de energia do banco elástico variando separadamente $\rho_S/\rho_M$, $c_L/c_M$ e $\eta$ (Fig. 26 da tese), com os máximos
locais (proeminência de pelo menos $0{,}1$ em $\log_{10}W_T$) marcados. Em cada painel, escolhem-se no banco as três curvas mais
próximas dos alvos, em distância logarítmica com peso $30$ nas grandezas que ficam fixas, entre as curvas com $1\le\eta\le10$; a soma em
$\ell$ vai até $\ell=12$.

In [ ]:
# casos reais do banco elástico mais próximos dos alvos (distância em log10, com pesos), como em visco_top.ipynb
BANCO_EL = RESUMO["elastico"].loc[(RESUMO["elastico"]["eta"] >= 1) & (RESUMO["elastico"]["eta"] <= 10)]
VARIACOES = {
    "densidade": dict(var="rho_ratio", alvos=[0.1, 1.0, 10.0], fixo=dict(c_razao=1.0, eta=2.0), pesos=dict(c_razao=30, rho_ratio=1, eta=30),
                      rot=lambda r: rf"$\rho_S/\rho_M={fmt_sig(r.rho_ratio)}$"),
    "velocidade_longitudinal": dict(var="c_razao", alvos=[0.1, 1.0, 10.0], fixo=dict(rho_ratio=1.0, eta=2.0),
                                    pesos=dict(c_razao=1, rho_ratio=30, eta=30), rot=lambda r: rf"$c_L/c_M={fmt_sig(r.c_razao)}$"),
    "eta": dict(var="eta", alvos=[1.16, 2.0, 5.0], fixo=dict(c_razao=1.0, rho_ratio=1.0), pesos=dict(c_razao=30, rho_ratio=30, eta=1),
                rot=lambda r: rf"$\eta={fmt_sig(r.eta, 3)}$"),
}
x_banco = np.unique(np.concatenate([np.geomspace(1e-4, 0.1, 1500, endpoint=False), np.linspace(0.1, 1.0, 1500)]))
for nome, v in VARIACOES.items():
    usados, casos = set(), []
    for alvo in v["alvos"]:
        ref = {**v["fixo"], v["var"]: alvo}
        d = BANCO_EL.loc[~BANCO_EL["_indice"].isin(usados)]
        dist = sum(v["pesos"][k] * (np.log10(d[k]) - np.log10(ref[k])) ** 2 for k in ref)
        r = d.loc[dist.idxmin()]
        usados.add(r["_indice"])
        casos.append(r)
    curvas = [energia_esfera(x_banco, "elastico", r.cL0, r.cT0, r.rhoS, lmax=12, normalizacao="ZS") for r in casos]
    n = figura_curvas(x_banco, curvas, [v["rot"](r) for r in casos], f"WT_variacao_{nome}", (0.0, 1.0),
                      [0.0, 0.25, 0.50, 0.75, 1.00], (1e-10, 1e8), [1e-10, 1e-4, 1e2, 1e8], casas=2, picos=True)
    for r, k in zip(casos, n):
        print(f"{nome:24s} curva {int(r['_indice']):5d}: c_L/c_M={r.c_razao:.3g}, rho_S/rho_M={r.rho_ratio:.3g}, eta={r.eta:.3g} | "
              f"{k} máximos marcados | N_res do banco = {int(r.N)}")

## 7. Mapa de estados ressonantes

Número de ressonâncias até $x_M=1$ no plano $(c_L/c_M,\ \rho_S/\rho_M)$, em escala logarítmica, com a mesma estética do mapa da
esfera fluida: cada célula mostra o maior número de ressonâncias entre as curvas que ela contém, e células vazias recebem o valor do
vizinho mais próximo. Como o banco também varia $\eta=c_L/c_T$, há um mapa para cada faixa de $\eta$:
$[2/\sqrt3;\,2)$, $[2;\,5)$ e $[5;\,\infty)$. Nos modelos viscoelásticos, os eixos e $\eta$ são os das velocidades de referência,
e cada célula reúne todos os valores de $\tilde\tau$ sorteados. A fronteira $P_{\mathrm{res}}=0{,}5$ do caso elástico vem da relação de
existência (Seção 8), avaliada no $\eta$ mediano da faixa, por isso esses mapas são desenhados depois do ajuste; os mapas de
Kelvin–Voigt e Maxwell mostram só a fronteira observada.

In [ ]:
LARGURA_BIN = 0.25
CELL_LINECOLOR, CELL_LINEWIDTH = "white", 0.42
LIMITES_MAPA = {"elastico": ((-3.0, 3.5), (-3.0, 3.0)), "voigt": ((-3.0, 3.5), (-3.0, 3.0)), "maxwell": ((-3.0, 3.5), (-4.0, 3.0))}
ETA_FAIXAS = [(2 / np.sqrt(3), 2.0), (2.0, 5.0), (5.0, np.inf)]
ROT_FAIXA = [r"$\eta\in[2/\sqrt{3};\,2)$", r"$\eta\in[2;\,5)$", r"$\eta\in[5;\,\infty)$"]


class Grade:
    def __init__(self, log_x, log_y):
        self.LOG_X, self.LOG_Y = log_x, log_y
        self.NX = int(round((log_x[1] - log_x[0]) / LARGURA_BIN))
        self.NY = int(round((log_y[1] - log_y[0]) / LARGURA_BIN))

    def celula_x(self, v):
        return (np.log10(v) - self.LOG_X[0]) / LARGURA_BIN - 0.5

    def celula_y(self, v):
        return (np.log10(v) - self.LOG_Y[0]) / LARGURA_BIN - 0.5


def mapa_binado(df, gr):
    ex, ey = np.linspace(*gr.LOG_X, gr.NX + 1), np.linspace(*gr.LOG_Y, gr.NY + 1)
    lx, ly, n = np.log10(df["c_razao"].to_numpy()), np.log10(df["rho_ratio"].to_numpy()), df["N"].to_numpy(float)
    ix = np.clip(np.searchsorted(ex, lx, side="right") - 1, 0, gr.NX - 1)
    iy = np.clip(np.searchsorted(ey, ly, side="right") - 1, 0, gr.NY - 1)
    contagem, maximo = np.zeros((gr.NX, gr.NY), int), np.full((gr.NX, gr.NY), -np.inf)
    np.add.at(contagem, (ix, iy), 1)
    np.maximum.at(maximo, (ix, iy), n)
    vazio = contagem == 0
    if vazio.any():
        ga, gb = np.meshgrid(0.5 * (ex[:-1] + ex[1:]), 0.5 * (ey[:-1] + ey[1:]), indexing="ij")
        _, viz = cKDTree(np.column_stack([lx, ly])).query(np.column_stack([ga[vazio], gb[vazio]]))
        maximo[vazio] = n[viz]
    return np.maximum(maximo, 0.0), contagem


def segmentos_fronteira(mask):
    seg = []
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            if not mask[i, j]:
                continue
            xl, xr, yb, yt = i - 0.5, i + 0.5, j - 0.5, j + 0.5
            if i == 0 or not mask[i - 1, j]: seg.append(((xl, yb), (xl, yt)))
            if i == mask.shape[0] - 1 or not mask[i + 1, j]: seg.append(((xr, yb), (xr, yt)))
            if j == 0 or not mask[i, j - 1]: seg.append(((xl, yb), (xr, yb)))
            if j == mask.shape[1] - 1 or not mask[i, j + 1]: seg.append(((xl, yt), (xr, yt)))
    return seg


def plot_mapa_fase(df, coef_exist, idioma, modelo, faixa, eta_ref=None, mostrar=True):
    pt = idioma == "pt"
    sep = VIRG if pt else "."
    gr = Grade(*LIMITES_MAPA[modelo])
    NX, NY = gr.NX, gr.NY
    mapa, contagem = mapa_binado(df, gr)
    n_max = max(int(np.ceil(mapa.max())), 1)
    norm = Normalize(vmin=0.0, vmax=np.log10(1.0 + n_max))
    ticks = np.unique(np.array([0] + [10 ** k for k in range(1, 6) if 10 ** k < 0.8 * n_max] + [n_max], dtype=int))
    with plt.rc_context({"font.family": "sans-serif"}):
        altura = 0.70
        largura = altura * NX / NY
        fig = plt.figure(figsize=(10.5, 10.5))
        ax = fig.add_axes([0.12, 0.13, largura, altura])
        cax = fig.add_axes([0.12 + largura + 0.02, 0.13, 0.03, altura])
        for i in range(NX):
            for j in range(NY):
                ax.add_patch(Rectangle((i - 0.5, j - 0.5), 1.0, 1.0, facecolor=RESONANCE_CMAP(norm(np.log10(1.0 + mapa[i, j]))),
                                       edgecolor=CELL_LINECOLOR, linewidth=CELL_LINEWIDTH, zorder=1))
        for a, b in segmentos_fronteira(mapa > 0):
            ax.plot([a[0], b[0]], [a[1], b[1]], solid_capstyle="butt", zorder=17, **FRONT_OBS)
        handles = [Line2D([0], [0], label="Fronteira observada" if pt else "Observed frontier", **FRONT_OBS)]
        if coef_exist is not None:
            GX, GY = np.meshgrid(np.linspace(*gr.LOG_X, 600), np.linspace(*gr.LOG_Y, 600))
            G = (termos_existencia(-GX.ravel() * np.log(10), GY.ravel() * np.log(10), np.full(GX.size, np.log(eta_ref)))
                 @ coef_exist).reshape(GX.shape)
            ax.contour(gr.celula_x(10 ** GX), gr.celula_y(10 ** GY), G, levels=[0.0], colors=[FRONT_LOG["color"]],
                       linewidths=FRONT_LOG["linewidth"], linestyles=[FRONT_LOG["linestyle"]], alpha=FRONT_LOG["alpha"], zorder=18)
            handles.append(Line2D([0], [0], label=("Fronteira logística\n" + r"($P_{\mathrm{res}}=0" + sep + "5$)") if pt
                                  else "Logistic frontier\n" + r"($P_{\mathrm{res}}=0.5$)", **FRONT_LOG))
        fmt = lambda t: f"{t:.4f}".rstrip("0").rstrip(".").replace(".", sep) if t < 1 else f"{t:g}"
        tx = 10.0 ** np.arange(np.ceil(gr.LOG_X[0]), gr.LOG_X[1] + 1e-9)
        ty = 10.0 ** np.arange(np.ceil(gr.LOG_Y[0]), gr.LOG_Y[1] + 1e-9)
        ax.set_xticks(gr.celula_x(tx)); ax.set_xticklabels([fmt(t) for t in tx])
        ax.set_yticks(gr.celula_y(ty)); ax.set_yticklabels([fmt(t) for t in ty])
        ax.set_xlabel(r"$c_L/c_M$", fontsize=23, labelpad=9)
        ax.set_ylabel(r"$\rho_S/\rho_M$", fontsize=23, labelpad=10)
        ax.set_xlim(-0.5, NX - 0.5); ax.set_ylim(-0.5, NY - 0.5)
        ax.set_aspect("equal", adjustable="box")
        leg = ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.985, 0.85), bbox_transform=ax.transAxes,
                        fontsize=20.0, frameon=True, framealpha=1.0, facecolor="white", edgecolor="none", fancybox=False,
                        borderpad=0.20, handlelength=2.4, handletextpad=0.8, labelspacing=0.45, borderaxespad=0.0)
        leg.set_zorder(100)
        rot_faixa = ROT_FAIXA[faixa].replace(";", ";" if pt else ",")
        ax.text(0.955, 0.965, rot_faixa + r"\quad $N_{\mathrm{res}}$", transform=ax.transAxes, fontsize=24, color="white",
                va="top", ha="right", bbox=dict(facecolor=BOX_COLOR, edgecolor="none", alpha=1.0, boxstyle="square,pad=0.14"), zorder=111)
        mapper = cm.ScalarMappable(norm=norm, cmap=RESONANCE_CMAP)
        mapper.set_array([])
        cbar = fig.colorbar(mapper, cax=cax, ticks=np.log10(1.0 + ticks))
        cbar.ax.set_yticklabels([str(t) for t in ticks])
        cbar.set_label("Ressonâncias" if pt else "Resonances", fontsize=20, labelpad=18, rotation=270, color="black")
        for eixo in (ax, cbar.ax):
            eixo.tick_params(axis="both", labelsize=15, width=1.25, length=4.5, direction="out")
            for spine in eixo.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(BOX_COLOR)
                spine.set_linewidth(1.8)
        nome = f"mapa_fase_{modelo}_eta{faixa + 1}_{idioma}"
        for ext in ("png", "pdf", "eps"):
            fig.savefig(os.path.join(FIG_DIR, f"{nome}.{ext}"), dpi=600, bbox_inches="tight")
        if mostrar:
            plt.show()
        plt.close(fig)


def fatia_eta(df, k):
    lo, hi = ETA_FAIXAS[k]
    return df.loc[(df["eta"] >= lo * (1 - 1e-9)) & (df["eta"] < hi)]


for m in ("voigt", "maxwell"):
    for k in range(len(ETA_FAIXAS)):
        d = fatia_eta(RESUMO[m], k)
        print(f"{NOME_MODELO[m]}, faixa {k + 1}: {len(d)} curvas, {100 * d['ressonante'].mean():.1f} % com ressonância")
        if len(d):
            for idioma in ("pt", "en"):
                plot_mapa_fase(d, None, idioma, m, k, mostrar=idioma == "pt")

## 8. Relação de existência (caso elástico)

Pela condição de ressonância (Seção 9), as raízes caem em $x_M=(c_L/c_M)\,\zeta$, e $\zeta$ depende só de $\ell$, de
$\rho=\rho_S/\rho_M$ e de $\eta$: os ramos longitudinal e transversal ficam acessíveis em $x_M\le1$ quando $c_M/c_L$ e
$c_M/c_T=\eta\,c_M/c_L$ são grandes o suficiente, com um limiar que depende de $\rho$. A relação de existência usa, por isso, as três
variáveis logarítmicas

$$
u=\ln\frac{c_M}{c_L},\qquad v=\ln\frac{\rho_S}{\rho_M},\qquad w=\ln\eta,
$$

na mesma forma quadrática da esfera fluida (que tinha só $u$ e $v$):

$$
P_{\mathrm{res}}(u,v,w)=\frac{1}{1+\exp[-g(u,v,w)]},\qquad
g=c_0+c_1u+c_2v+c_3w+c_4u^2+c_5uv+c_6uw+c_7v^2+c_8vw+c_9w^2 .
$$

Há ressonância em $10^{-4}\le x_M\le1$ quando $P_{\mathrm{res}}\ge1/2$, isto é, $g\ge0$. Todos os coeficientes são ajustados ao banco
elástico por regressão logística (scikit-learn, penalização L2 padrão); $20\,\%$ das curvas, sorteadas uma vez (semente 42), ficam para
o teste e são as mesmas curvas de teste da recorrência. As incertezas vêm da informação de Fisher.

In [ ]:
TEST_FRACTION, RANDOM_SEED = 0.20, 42
TEX_TERMOS = [r"$1$", r"$u$", r"$v$", r"$w$", r"$u^2$", r"$u\,v$", r"$u\,w$", r"$v^2$", r"$v\,w$", r"$w^2$"]
NOMES_TERMOS = ["1", "u", "v", "w", "u^2", "u v", "u w", "v^2", "v w", "w^2"]


def termos_existencia(u, v, w):
    # [1, u, v, w, u^2, u v, u w, v^2, v w, w^2] (mesma ordem de PolynomialFeatures)
    u, v, w = np.asarray(u, float), np.asarray(v, float), np.asarray(w, float)
    return np.column_stack([np.ones_like(u), u, v, w, u * u, u * v, u * w, v * v, v * w, w * w])


def p_res(u, v, w, coef):
    return 1.0 / (1.0 + np.exp(-np.clip(termos_existencia(u, v, w) @ coef, -700, 700)))


resumo = RESUMO["elastico"]
rng = np.random.default_rng(RANDOM_SEED)
indices = np.arange(len(resumo))
rng.shuffle(indices)
n_teste = int(round(TEST_FRACTION * len(indices)))
treino_c, teste_c = resumo.iloc[indices[n_teste:]].copy(), resumo.iloc[indices[:n_teste]].copy()
GLOBAIS_TESTE = set(teste_c["_indice"])

modelo = Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)), ("scale", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=5000))]).fit(treino_c[["u", "v", "w"]].to_numpy(), treino_c["ressonante"].to_numpy())
esc, lr = modelo.named_steps["scale"], modelo.named_steps["clf"]
COEF_EXIST = np.r_[lr.intercept_[0] - np.sum(lr.coef_[0] * esc.mean_ / esc.scale_), lr.coef_[0] / esc.scale_]
X = termos_existencia(treino_c["u"], treino_c["v"], treino_c["w"])
p = p_res(treino_c["u"], treino_c["v"], treino_c["w"], COEF_EXIST)
ERRO_EXIST = np.sqrt(np.clip(np.diag(np.linalg.pinv(X.T @ (X * (p * (1 - p))[:, None]))), 0, None))


def metricas_existencia(df):
    y, p = df["ressonante"].to_numpy(), p_res(df["u"], df["v"], df["w"], COEF_EXIST)
    pred = (p >= 0.5).astype(int)
    return {"N": len(y), "fracao": y.mean(), "acuracia": accuracy_score(y, pred), "precisao": precision_score(y, pred, zero_division=0),
            "revocacao": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0),
            "AUC": roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan,
            "FP": int(((pred == 1) & (y == 0)).sum()), "FN": int(((pred == 0) & (y == 1)).sum())}


MET_EXIST = {"treino": metricas_existencia(treino_c), "teste": metricas_existencia(teste_c)}
for i, t in enumerate(NOMES_TERMOS):
    aviso = "" if abs(COEF_EXIST[i]) >= 2 * ERRO_EXIST[i] else "   <- |c| < 2 sigma (mantido na fórmula)"
    print(f"c{i} ({t:>5s}) = {COEF_EXIST[i]:+.5e} ± {ERRO_EXIST[i]:.2e}{aviso}")
for parte, m_ in MET_EXIST.items():
    print(parte, {k: round(v, 4) if isinstance(v, float) else v for k, v in m_.items()})

### Tabelas de coeficientes e de desempenho

Mesmo formato das tabelas da tese: valor $\pm$ incerteza com dois algarismos significativos, em PNG, PDF, SVG, CSV e `.tex`.

In [ ]:
salvar_tabela_coeficientes("tabela_coeficientes_existencia", [f"$c_{{{i}}}$  " + TEX_TERMOS[i] for i in range(len(TEX_TERMOS))],
                           ["Valor"], [[c] for c in COEF_EXIST], [[e] for e in ERRO_EXIST], figsize=(4.6, 6.2), first_col_width=0.45)

with open(os.path.join(TAB_DIR, "tabela_tese_existencia.tex"), "w", encoding="utf-8") as fh:
    fh.write("\\begin{tabular}{lc}\n\\toprule\n\\textbf{Termo} & $c_i$ \\\\\n\\midrule\n")
    for i in range(len(TEX_TERMOS)):
        fh.write(f"{TEX_TERMOS[i]} & {fmt_pm(COEF_EXIST[i], ERRO_EXIST[i])} \\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n")
print(open(os.path.join(TAB_DIR, "tabela_tese_existencia.tex"), encoding="utf-8").read())

rot = {"N": "Curvas", "fracao": "Fração ressonante", "acuracia": "Acurácia", "precisao": "Precisão", "revocacao": "Revocação",
       "F1": r"$F_1$", "AUC": "ROC AUC", "FP": "Falsos positivos", "FN": "Falsos negativos"}
fmt = lambda k, v: f"${milhar(v)}$" if k in ("N", "FP", "FN") else f"${dec(v, 4 if k == 'AUC' else 3)}$"
salvar_tabela_texto("tabela_metricas_existencia", list(rot.values()), ["Treino", "Teste"],
                    [[fmt(k, MET_EXIST[p][k]) for p in ("treino", "teste")] for k in rot], pd.DataFrame(MET_EXIST),
                    figsize=(5.6, 4.2), first_col_width=0.50)

### Desempenho no teste e mapa de estados

In [ ]:
y = teste_c["ressonante"].to_numpy(float)
p = p_res(teste_c["u"], teste_c["v"], teste_c["w"], COEF_EXIST)
xj = y + np.random.default_rng(RANDOM_SEED).uniform(-0.05, 0.05, len(y))
m_ = MET_EXIST["teste"]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(xj[y == 0], p[y == 0], s=14, facecolors="white", edgecolors="#6488A5", linewidths=0.7, zorder=2, label="Sem ressonância")
ax.scatter(xj[y == 1], p[y == 1], s=14, facecolors="#8B1A1A", edgecolors="black", linewidths=0.4, zorder=3, label="Com ressonância")
ax.axhline(0.5, color="black", linestyle=":", linewidth=1.2, zorder=1, label=r"$P_{\mathrm{res}}=0{,}5$")
ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-0.03, 1.03)
ax.set_xticks([0, 1])
ax.set_xticklabels([r"$0$", r"$1$"])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels([r"$0$", r"$0{,}25$", r"$0{,}50$", r"$0{,}75$", r"$1$"])
estilizar(ax, "Existência observada", r"$P_{\mathrm{res}}$ previsto")
adicionar_caixa(ax, f"$N={milhar(m_['N'])}$\n" + f"Acurácia $={dec(m_['acuracia'], 3)}$\n" + f"Precisão $={dec(m_['precisao'], 3)}$\n"
                + f"Revocação $={dec(m_['revocacao'], 3)}$\n" + f"$F_1={dec(m_['F1'], 3)}$\n" + f"ROC AUC $={dec(m_['AUC'], 4)}$",
                x=0.035, ha="left")
adicionar_legenda(ax)
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_existencia")
plt.show()
plt.close(fig)

for k in range(len(ETA_FAIXAS)):
    d = fatia_eta(resumo, k)
    eta_ref = float(np.median(d["eta"])) if len(d) else float(np.sqrt(ETA_FAIXAS[k][0] * min(ETA_FAIXAS[k][1], 50)))
    yk = d["ressonante"].to_numpy()
    pk = p_res(d["u"], d["v"], d["w"], COEF_EXIST) >= 0.5
    print(f"faixa {k + 1}: {len(d)} curvas | eta mediano {eta_ref:.3g} | concordância P_res >= 0,5 com o observado: {np.mean(pk == yk):.4f}")
    if len(d):
        for idioma in ("pt", "en"):
            plot_mapa_fase(d, COEF_EXIST, idioma, "elastico", k, eta_ref=eta_ref, mostrar=idioma == "pt")

### Comparação com a forma anterior ($u_L$, $u_R$)

A forma usada antes em `visco_top.ipynb` escreve a existência com duas variáveis físicas: $u_L=\ln\big[(c_M/c_L)/w_0(\rho_S/\rho_M)\big]$,
com $w_0(r)$ o primeiro zero de $z\,j_0'(z)+r\,j_0(z)$, isto é, de $(1-r)\sin z=z\cos z$ (primeira ressonância longitudinal de
$\ell=0$ em $x_M\le1$ quando $u_L\ge0$), e $u_R=\ln\big[(c_M/c_R)/(5/2)\big]$, com $c_R$ a velocidade de Rayleigh do sólido (modo de
superfície de $\ell=2$ em $x_M\simeq(\ell+1/2)\,c_R/c_M\le1$ quando $u_R\ge0$). A regra sem coeficientes é $u_L\ge0$ ou $u_R\ge0$, e a
logística é $g=c_0+c_1u_L+c_2u_L^2+c_3u_R+c_4u_R^2+c_5\ln\alpha+c_6(\ln\alpha)^2+c_7u_Lu_R$. Aqui ela é ajustada do mesmo jeito que a
relação principal (scikit-learn, L2 padrão, mesmas curvas de treino e de teste), para que as métricas sejam comparáveis.

In [ ]:
def w0_zero(r):
    # primeiro zero positivo de (1 - r) sin z - z cos z, vetorizado (bissecção em (0, pi])
    r = np.asarray(r, float)
    a, b = np.full(r.shape, 1e-9), np.full(r.shape, np.pi)
    f = lambda z: (1.0 - r) * np.sin(z) - z * np.cos(z)
    for _ in range(80):
        m = 0.5 * (a + b)
        neg = f(m) < 0
        a, b = np.where(neg, m, a), np.where(neg, b, m)
    return 0.5 * (a + b)


def cR_sobre_cT(eta):
    # raiz s = (c_R/c_T)^2 da equação de Rayleigh, (2 - s)^2 = 4 sqrt(1 - s) sqrt(1 - s/eta^2), vetorizada
    eta = np.asarray(eta, float)
    a, b = np.full(eta.shape, 1e-9), np.full(eta.shape, 1 - 1e-12)
    g = lambda s: (2 - s) ** 2 - 4 * np.sqrt(1 - s) * np.sqrt(1 - s / eta ** 2)
    for _ in range(80):
        m = 0.5 * (a + b)
        neg = g(m) < 0
        a, b = np.where(neg, m, a), np.where(neg, b, m)
    return np.sqrt(0.5 * (a + b))


TEX_TERMOS_R = [r"$1$", r"$u_L$", r"$u_L^2$", r"$u_R$", r"$u_R^2$", r"$\ln\alpha$", r"$(\ln\alpha)^2$", r"$u_Lu_R$"]


def variaveis_R(beta, rho_ratio, eta):
    beta, rho_ratio, eta = (np.asarray(v, float) for v in (beta, rho_ratio, eta))
    return np.log(beta / w0_zero(rho_ratio)), np.log(beta * eta / cR_sobre_cT(eta) / 2.5)


def termos_R(uL, uR, la):
    return np.column_stack([np.ones_like(uL), uL, uL ** 2, uR, uR ** 2, la, la ** 2, uL * uR])


for df in (treino_c, teste_c):
    df["uL"], df["uR"] = variaveis_R(df["beta"], df["rho_ratio"], df["eta"])
    df["la"] = np.log(df["alpha"])
XR = termos_R(treino_c["uL"], treino_c["uR"], treino_c["la"])
esc_R = StandardScaler().fit(XR[:, 1:])
lr_R = LogisticRegression(max_iter=5000).fit(esc_R.transform(XR[:, 1:]), treino_c["ressonante"])
COEF_R = np.r_[lr_R.intercept_[0] - np.sum(lr_R.coef_[0] * esc_R.mean_ / esc_R.scale_), lr_R.coef_[0] / esc_R.scale_]
pR = 1 / (1 + np.exp(-np.clip(XR @ COEF_R, -700, 700)))
ERRO_R = np.sqrt(np.clip(np.diag(np.linalg.pinv(XR.T @ (XR * (pR * (1 - pR))[:, None]))), 0, None))
for i, t in enumerate(["1", "u_L", "u_L^2", "u_R", "u_R^2", "ln alpha", "(ln alpha)^2", "u_L u_R"]):
    print(f"c{i} ({t:>12s}) = {COEF_R[i]:+.5e} ± {ERRO_R[i]:.2e}")


def metricas_bin(y, pred, score=None):
    return {"acuracia": accuracy_score(y, pred), "precisao": precision_score(y, pred, zero_division=0),
            "revocacao": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0),
            "AUC": roc_auc_score(y, score) if score is not None and 0 < y.sum() < len(y) else np.nan}


yt = teste_c["ressonante"].to_numpy()
sR = termos_R(teste_c["uL"], teste_c["uR"], teste_c["la"]) @ COEF_R
COMP_EXIST = {"Regra $u_L\\geq0$ ou $u_R\\geq0$": metricas_bin(yt, ((teste_c["uL"] >= 0) | (teste_c["uR"] >= 0)).to_numpy().astype(int)),
              "Logística em $u_L$, $u_R$": metricas_bin(yt, (sR >= 0).astype(int), sR),
              "Logística em $u$, $v$, $w$": {k: MET_EXIST["teste"][k] for k in ("acuracia", "precisao", "revocacao", "F1", "AUC")}}
rot = {"acuracia": "Acurácia", "precisao": "Precisão", "revocacao": "Revocação", "F1": r"$F_1$", "AUC": "ROC AUC"}
salvar_tabela_texto("tabela_comparacao_existencia", list(rot.values()), [c.replace(", $u_R$", ",\n$u_R$").replace(", $v$, $w$", ",\n$v$, $w$")
                                                                       for c in COMP_EXIST],
                    [["--" if not np.isfinite(COMP_EXIST[c][k]) else f"${dec(COMP_EXIST[c][k], 4 if k == 'AUC' else 3)}$"
                      for c in COMP_EXIST] for k in rot], pd.DataFrame(COMP_EXIST), figsize=(8.6, 3.0), first_col_width=0.25)
salvar_tabela_coeficientes("tabela_coeficientes_existencia_uL_uR", [f"$c_{{{i}}}$  " + TEX_TERMOS_R[i] for i in range(8)],
                           ["Valor"], [[c] for c in COEF_R], [[e] for e in ERRO_R], figsize=(4.6, 5.2), first_col_width=0.45)

# mapas de P_res da forma u_L, u_R para tres valores de eta, com os pontos ressonantes do banco
cl, rs = np.logspace(-3, 3, 160), np.logspace(-3, 3, 160)
CLg, RSg = np.meshgrid(cl, rs)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for k, (ax, e0) in enumerate(zip(axes, (1.16, 3.0, 10.0))):
    uLg, uRg = variaveis_R(1 / CLg.ravel(), RSg.ravel(), np.full(CLg.size, e0))
    P = (1 / (1 + np.exp(-np.clip(termos_R(uLg, uRg, np.log(1 / RSg.ravel())) @ COEF_R, -700, 700)))).reshape(CLg.shape)
    pc = ax.pcolormesh(CLg, RSg, P, cmap=RESONANCE_CMAP, vmin=0, vmax=1, shading="auto", rasterized=True, zorder=1)
    ax.contour(CLg, RSg, P, levels=[0.5], colors=[FRONT_LOG["color"]], linestyles=[FRONT_LOG["linestyle"]],
               linewidths=FRONT_LOG["linewidth"], zorder=3)
    ax.contour(CLg, RSg, ((uLg >= 0) | (uRg >= 0)).reshape(CLg.shape).astype(float), levels=[0.5], colors=[FRONT_OBS["color"]],
               linestyles=[FRONT_OBS["linestyle"]], linewidths=FRONT_OBS["linewidth"], zorder=3)
    d = resumo.loc[(np.abs(np.log10(resumo["eta"] / e0)) < 0.15) & (resumo["ressonante"] == 1)]
    ax.scatter(d["c_razao"], d["rho_ratio"], zorder=4, **MARCADOR_DADOS)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(1e-3, 1e3); ax.set_ylim(1e-3, 1e3)
    estilizar(ax, r"$c_L/c_M$", r"$\rho_S/\rho_M$" if k == 0 else None)
    adicionar_caixa(ax, rf"$\eta={fmt_sig(e0, 3)}$")
h = [Line2D([0], [0], label=r"$P_{\mathrm{res}}=0{,}5$", **FRONT_LOG), Line2D([0], [0], label="Regra sem coeficientes", **FRONT_OBS),
     Line2D([0], [0], linestyle="none", marker="o", markersize=3.5, markerfacecolor="white", markeredgecolor="0.45",
            label="Ressonante no banco")]
adicionar_legenda(axes[0], handles=h, loc="lower left")
cb = fig.colorbar(pc, ax=list(axes), pad=0.015, fraction=0.02)
cb.set_label(r"$P_{\mathrm{res}}$", fontsize=12, rotation=270, labelpad=16)
cb.ax.tick_params(labelsize=10, direction="in")
cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: dec(v, 2)))
salvar_figura(fig, "P_res_elastico_uL_uR")
plt.show()
plt.close(fig)

## 9. Relação de recorrência (caso elástico)

### 9.1 Equação característica no limite de baixa frequência

Os coeficientes $a_\ell$, $b_\ell$ e $s_\ell$ têm o mesmo denominador, cujos zeros coincidem com os de $\det[M]_\ell$. Expandindo o
determinante pela terceira coluna (Eq. 5.103),

$$
-\frac{x_Mh_\ell^{(1)\prime}(x_M)}{h_\ell^{(1)}(x_M)}\,E_\ell(z_L,z_T)+\rho_Mc_L^2z_L^2\,G_\ell(z_L,z_T)=0,
$$

$$
\begin{aligned}
E_\ell&=z_L^2\big[\lambda j_\ell(z_L)-2\mu j_\ell''(z_L)\big]\,T_\ell(z_T)+4\mu\,\ell(\ell+1)\big[z_Tj_\ell'(z_T)-j_\ell(z_T)\big]\big[z_Lj_\ell'(z_L)-j_\ell(z_L)\big],\\
G_\ell&=z_Lj_\ell'(z_L)\,T_\ell(z_T)-2\ell(\ell+1)\,j_\ell(z_T)\big[z_Lj_\ell'(z_L)-j_\ell(z_L)\big],\qquad
T_\ell(z_T)=z_T^2j_\ell''(z_T)+\big[\ell(\ell+1)-2\big]j_\ell(z_T).
\end{aligned}
$$

$E_\ell=0$ são os modos esferoidais da esfera livre, e o termo em $\rho_M$ é o carregamento do fluido. Para $x_M\ll1$,
$-x_Mh_\ell^{(1)\prime}/h_\ell^{(1)}\simeq\ell+1$ (Eq. 5.105), e a condição fica (Eq. 5.106)

$$
(\ell+1)\,E_\ell(z_L,\eta z_L)+\rho_Mc_L^2z_L^2\,G_\ell(z_L,\eta z_L)=0 ,
$$

que, dividida por $\rho_Sc_L^2$, depende só de $\ell$, de $\eta$ e de $\rho=\rho_S/\rho_M$ ($\lambda/\rho_Sc_L^2=1-2/\eta^2$,
$\mu/\rho_Sc_L^2=1/\eta^2$) e é real para $z_L$ real. Cada raiz $\zeta_{\ell,k}(\rho,\eta)$ dá uma posição

$$
x_{M;\ell,k}=\frac{c_L}{c_M}\,\zeta_{\ell,k}(\rho,\eta),
$$

sem busca no plano complexo. Para $\ell=0$ o fator comum $T_0(z_T)$ não tem significado físico e sai (Eq. 5.108):
$\big(1-\tfrac{2}{\eta^2}\big)j_0(z_L)-\tfrac{2}{\eta^2}j_0''(z_L)+\tfrac{\rho_M}{\rho_S}z_Lj_0'(z_L)=0$. No código, as
derivadas são escritas só com $j_\ell$ e $j_{\ell+1}$ ($zj_\ell'=\ell j_\ell-zj_{\ell+1}$ e
$z^2j_\ell''=(\ell^2-\ell-z^2)j_\ell+2zj_{\ell+1}$), o que evita o cancelamento em $z$ pequeno. As raízes saem da troca de sinal numa
malha com $16$ pontos por $\pi$ de fase acumulada, $(1+\eta)z_L$, e de bissecção; famílias com mais de $3\,000$ raízes por $\ell$ até
$x_M=1{,}2$ (sólidos muito lentos) ficam fora da recorrência.

### 9.2 Ramos longitudinal e transversal, espaçamento e contagem

Desprezando as parcelas de acoplamento proporcionais a $4\mu\ell(\ell+1)$ em $E_\ell$ e a $2\ell(\ell+1)$ em $G_\ell$, a condição se
fatora em dois ramos (Eqs. 5.110 a 5.114): o longitudinal,
$(\ell+1)\big[(1-\tfrac{2}{\eta^2})j_\ell(z_L)-\tfrac{2}{\eta^2}j_\ell''(z_L)\big]+\tfrac{\rho_M}{\rho_S}z_Lj_\ell'(z_L)=0$, que para $z_L$ grande
tende à condição de Robin da esfera fluida, e o transversal, $T_\ell(z_T)=0$, cujos zeros tendem aos de $j_\ell$. Com
$w_{\ell,n}(\rho,\eta)$ e $\tau_{\ell,n}$ os zeros de cada ramo, $x^{(L)}_{M,n}=\frac{c_L}{c_M}w_{\ell,n}$ e $x^{(T)}_{M,n}=\frac{c_T}{c_M}\tau_{\ell,n}$
(Eq. 5.116), e pela expansão de McMahon os espaçamentos tendem a

$$
\Delta_L\to\frac{\pi c_L}{c_M},\qquad \Delta_T\to\frac{\pi c_T}{c_M},\qquad
N_\ell(x_M)=\frac{x_M}{\pi}\left(\frac{c_M}{c_L}+\frac{c_M}{c_T}\right)+O(1),
$$

de modo que, com os dois ramos intercalados, o espaçamento médio entre ressonâncias consecutivas de um mesmo $\ell$ é
$\Delta_W=\pi\big/\big(c_M/c_L+c_M/c_T\big)$ (Eq. 5.120). Nas primeiras raízes de cada $\ell$ os ramos se misturam e as posições
devem sair diretamente da Eq. (5.106).

### 9.3 Recorrência

Por família $\ell$, a ressonância seguinte vem da raiz seguinte da Eq. (5.106): identifica-se a raiz $\zeta_{\ell,k}$ mais próxima
de $z_L=(c_M/c_L)\,x_{M,n}$ e

$$
x_{M,n+1}=x_{M,n}+S\,\frac{c_L}{c_M}\Big[\zeta_{\ell,k+1}(\rho,\eta)-\zeta_{\ell,k}(\rho,\eta)\Big].
$$

Quando os ramos se separam, raízes consecutivas do mesmo ramo reproduzem as Eqs. (5.117) e (5.118), e o passo tende a
$\pi c_L/c_M$ ou $\pi c_T/c_M$. O único coeficiente, o fator de escala $S$, é ajustado aos dados: em cada família de treino
$(\text{curva},\ell)$, $S_f$ é a mediana de $(x_{M,n+1}-x_{M,n})\big/\big[(c_L/c_M)(\zeta_{\ell,k+1}-\zeta_{\ell,k})\big]$ sobre as suas
transições; $S$ é a mediana dos $S_f$, e o erro da mediana vem de $200$ reamostragens das famílias (*bootstrap*). Avalia-se em
seguida, nas curvas de teste, a posição prevista de cada ressonância a partir da anterior.

Entram na recorrência só as curvas em que a contagem final veio dos polos (`modo_contagem = polos_area`): nelas cada ressonância é
um polo de $D_\ell$ com o seu $\ell$, e as famílias estão completas. Nas curvas contadas pela malha refinada, o $\ell$ de cada pico vem
do polo mais próximo e as famílias podem ter lacunas e picos repetidos; nas curvas normais não há $\ell$.

In [ ]:
PASSOS_POR_PI = 16            # pontos da malha por pi de fase acumulada, (1 + eta) z_L
RAIZES_MAX = 3000             # fase/pi acima disso: família fora da recorrência (sólido lento demais)
Z_FATOR = 1.2                 # raízes até x_M = 1,2 (a seguinte da última ressonância observada)


def termos_caracteristica(l, z, rho, eta, J=None, JT=None):
    # Eq. (5.106) dividida por rho_S c_L^2 (l >= 1) ou Eq. (5.108) vezes z^2 (l = 0), só com j_l e j_{l+1}
    eps = 1.0 / eta ** 2
    mu = z ** 2 / rho                               # (rho_M/rho_S) z_L^2
    zT = eta * z
    jl = spherical_jn(l, z) if J is None else J[l]
    pl = spherical_jn(l + 1, z) if J is None else J[l + 1]
    AL = (z ** 2 - 2 * eps * (l * l - l)) * jl - 4 * eps * z * pl      # z^2 [(1 - 2 eps) j - 2 eps j'']
    dL = l * jl - z * pl                                               # z j'
    if l == 0:
        return (AL, mu * dL)
    jt = spherical_jn(l, zT) if JT is None else JT[l]
    pt = spherical_jn(l + 1, zT) if JT is None else JT[l + 1]
    T = (2 * l * l - 2 - zT ** 2) * jt + 2 * zT * pt                   # T_l(z_T)
    BT, BL = (l - 1) * jt - zT * pt, (l - 1) * jl - z * pl             # z_T j' - j  e  z_L j' - j
    L = l * (l + 1)
    return ((l + 1) * AL * T, (l + 1) * 4 * eps * L * BT * BL, mu * dL * T, -2 * L * mu * jt * BL)


def F_caracteristica(l, z, rho, eta):
    return sum(termos_caracteristica(l, z, rho, eta))


def raizes_caracteristica(rho, eta, z_max, ls, it=50):
    # raízes reais zeta_{l,k}(rho, eta) em (0, z_max]: troca de sinal na malha + bissecção; None se houver raízes demais
    n = int(PASSOS_POR_PI * (1 + eta) * z_max / np.pi)
    if n > PASSOS_POR_PI * RAIZES_MAX:
        return None
    z = np.unique(np.concatenate([np.geomspace(1e-4 * z_max, min(1.0, z_max), 400), np.linspace(0, z_max, max(n, 50))[1:]]))
    ordens = np.arange(max(ls) + 2)[:, None]
    J, JT = spherical_jn(ordens, z[None, :]), spherical_jn(ordens, eta * z[None, :])
    saida = {}
    for l in ls:
        t = termos_caracteristica(l, z, rho, eta, J, JT)
        f, escala = sum(t), sum(np.abs(u) for u in t)
        bom = np.abs(f) > 1e-12 * escala                     # fora do cancelamento por arredondamento
        i = np.flatnonzero(bom[:-1] & bom[1:] & (np.sign(f[:-1]) * np.sign(f[1:]) < 0))
        a, b, fa = z[i], z[i + 1], f[i]
        for _ in range(it):
            mid = 0.5 * (a + b)
            fm = F_caracteristica(l, mid, rho, eta)
            s = np.sign(fm) == np.sign(fa)
            a, fa, b = np.where(s, mid, a), np.where(s, fm, fa), np.where(s, b, mid)
        saida[l] = 0.5 * (a + b)
    return saida


# famílias observadas: ressonâncias com o l do polo associado, em 1e-4 <= x_M <= 1
PAR = RESUMO["elastico"].set_index("_indice")
CURVAS_POLOS = set(PAR.index[PAR["modo_contagem"] == "polos_area"])     # contagem final pelos polos: famílias completas
FAM = PICOS["elastico"].loc[np.isfinite(PICOS["elastico"]["l"])].copy()
FAM["l"] = FAM["l"].astype(int)
FAM = FAM.loc[FAM["curva"].isin(CURVAS_POLOS)].sort_values(["curva", "l", "x"]).reset_index(drop=True)
print(f"curvas contadas pelos polos: {len(CURVAS_POLOS)} de {int(PAR['ressonante'].sum())} curvas ressonantes")
tam = FAM.groupby(["curva", "l"]).size()
LS_CURVA = FAM.groupby("curva")["l"].unique()

t0 = time.perf_counter()
RAIZES, FORA = {}, []
for c, ls in LS_CURVA.items():
    beta, eta, rho = PAR.at[c, "beta"], PAR.at[c, "eta"], PAR.at[c, "rho_ratio"]
    R = raizes_caracteristica(rho, eta, Z_FATOR * beta, sorted(int(l) for l in ls))
    if R is None:
        FORA.append(c)
        continue
    RAIZES[c] = {l: r / beta for l, r in R.items()}           # em x_M
T_RAIZES = (time.perf_counter() - t0) / max(len(LS_CURVA), 1)
print(f"famílias: {len(tam)} em {len(LS_CURVA)} curvas ({int((tam >= 2).sum())} com duas ou mais ressonâncias)")
print(f"raízes da Eq. (5.106) em {len(RAIZES)} curvas ({T_RAIZES * 1e3:.3g} ms por curva); "
      f"{len(FORA)} curvas fora (mais de {RAIZES_MAX} raízes por l até x_M = {Z_FATOR})")

# validação: cada ressonância observada contra a raiz mais próxima da sua família
erro_pos, zT_pos = [], []
for (c, l), d in FAM.loc[FAM["curva"].isin(RAIZES)].groupby(["curva", "l"]):
    r = RAIZES[c][l]
    if len(r) == 0:
        continue
    x = d["x"].to_numpy()
    k = np.clip(np.searchsorted(r, x), 1, len(r) - 1) if len(r) > 1 else np.zeros(len(x), int)
    perto = r[k] if len(r) == 1 else np.where(np.abs(r[k - 1] - x) < np.abs(r[k] - x), r[k - 1], r[k])
    erro_pos.append(100 * np.abs(perto - x) / x)
    zT_pos.append(x * PAR.at[c, "beta"] * PAR.at[c, "eta"])
ERRO_POS, ZT_POS = np.concatenate(erro_pos), np.concatenate(zT_pos)
for nome, mk in ((r"z_T < pi", ZT_POS < np.pi), (r"z_T >= pi", ZT_POS >= np.pi), ("todas", np.ones(len(ZT_POS), bool))):
    if mk.any():
        print(f"posição, {nome:9s}: {mk.sum()} ressonâncias | erro mediano {np.median(ERRO_POS[mk]):.2g} % | "
              f"percentil 90 {np.quantile(ERRO_POS[mk], 0.9):.2g} %")

In [ ]:
def transicoes(fam):
    g, l, x = fam["curva"].to_numpy(), fam["l"].to_numpy(), fam["x"].to_numpy()
    mesmo = (g[1:] == g[:-1]) & (l[1:] == l[:-1])
    tr = pd.DataFrame({"curva": g[:-1][mesmo], "l": l[:-1][mesmo], "x_n": x[:-1][mesmo], "x_np1": x[1:][mesmo]})
    tr = tr.loc[tr["curva"].isin(RAIZES)].reset_index(drop=True)
    passo = np.full(len(tr), np.nan)
    for (c, L), idx in tr.groupby(["curva", "l"]).indices.items():
        r = RAIZES[c][L]
        if len(r) < 2:
            continue
        xn = tr["x_n"].to_numpy()[idx]
        k = np.clip(np.searchsorted(r, xn), 1, len(r) - 1)
        k = np.where(np.abs(r[k - 1] - xn) <= np.abs(r[k] - xn), k - 1, k)          # raiz mais próxima de x_n
        ok = k + 1 < len(r)
        p = np.full(len(idx), np.nan)
        p[ok] = r[k[ok] + 1] - r[k[ok]]
        passo[idx] = p
    tr["passo_car"] = passo
    tr = tr.join(PAR[["beta", "eta", "rho_ratio", "c_razao"]], on="curva")
    tr["passo_W"] = np.pi / (tr["beta"] * (1 + tr["eta"]))                        # Delta_W = pi/(c_M/c_L + c_M/c_T)
    tr["zT"] = tr["x_n"] * tr["beta"] * tr["eta"]                                  # z_T = (c_M/c_T) x_n
    tr["teste"] = tr["curva"].isin(GLOBAIS_TESTE)
    return tr


def ajusta_escala(tr, coluna):
    # S_f = mediana por família de (passo observado)/(passo da forma); S = mediana dos S_f; erro por bootstrap das famílias
    q = (tr["x_np1"] - tr["x_n"]) / tr[coluna]
    q = q[np.isfinite(q) & (q > 0)]
    Sf = q.groupby([tr.loc[q.index, "curva"], tr.loc[q.index, "l"]]).median()
    if len(Sf) == 0:
        return np.nan, np.nan, Sf
    v = Sf.to_numpy()
    boot = [np.median(np.random.default_rng(RANDOM_SEED + k).choice(v, len(v))) for k in range(200)]
    return float(np.median(v)), float(np.std(boot)), Sf


TR = transicoes(FAM)
TR_TREINO, TR_TESTE = TR.loc[~TR["teste"]], TR.loc[TR["teste"]]
S_CAR, ERRO_S_CAR, SF_CAR = ajusta_escala(TR_TREINO, "passo_car")
S_W, ERRO_S_W, SF_W = ajusta_escala(TR_TREINO, "passo_W")
print(f"transições: {len(TR)} ({len(TR_TREINO)} de treino, {len(TR_TESTE)} de teste)")
print(f"S (equação característica) = {S_CAR:.5f} ± {ERRO_S_CAR:.5f}  ({len(SF_CAR)} famílias de treino)")
print(f"S_W (espaçamento médio)    = {S_W:.5f} ± {ERRO_S_W:.5f}  ({len(SF_W)} famílias de treino)")

# S por ordem multipolar, como a Tabela 3 da tese faz por fator de mistura
por_l = []
for l in range(L_MAX + 1):
    s, e, sf = ajusta_escala(TR_TREINO.loc[TR_TREINO["l"] == l], "passo_car")
    por_l.append((l, s, e, len(sf)))
S_POR_L = pd.DataFrame(por_l, columns=["l", "S", "erro", "familias"]).set_index("l")
print(S_POR_L.round(4).to_string())

linhas = [r"Todos os $\ell$"] + [rf"$\ell={l}$" for l in S_POR_L.index if S_POR_L.at[l, "familias"] > 0]
dados = [[fmt_pm(S_CAR, ERRO_S_CAR), f"${milhar(len(SF_CAR))}$"]] + \
        [[fmt_pm(S_POR_L.at[l, "S"], S_POR_L.at[l, "erro"]), f"${milhar(S_POR_L.at[l, 'familias'])}$"]
         for l in S_POR_L.index if S_POR_L.at[l, "familias"] > 0]
valores = pd.concat([pd.DataFrame({"S": [S_CAR], "erro": [ERRO_S_CAR], "familias": [len(SF_CAR)]}, index=["todos"]), S_POR_L])
salvar_tabela_texto("tabela_S_recorrencia", linhas, [r"$S$", "Famílias"], dados, valores,
                    figsize=(5.4, 0.42 * (len(linhas) + 1) + 0.3), first_col_width=0.36, header_label="Ordem")

In [ ]:
FORMAS = {"car_S": ("passo_car", S_CAR), "car": ("passo_car", 1.0), "W": ("passo_W", S_W)}
NOMES_FORMA = {"car_S": r"Equação característica, $S$ ajustado", "car": r"Equação característica, $S=1$",
               "W": r"Espaçamento médio $\Delta_W$, $S_W$ ajustado"}
NOMES_FORMA_TXT = {"car_S": "eq. característica, S ajustado", "car": "eq. característica, S = 1", "W": "Delta_W, S_W ajustado"}
ESTILO = {"car_S": dict(color="#8B1A1A", linestyle="-", linewidth=2.0), "car": dict(color="#8B1A1A", linestyle="--", linewidth=2.0),
          "W": dict(color="#6488A5", linestyle="-.", linewidth=1.75)}


def r2_mape(obs, pred):
    ok = np.isfinite(obs) & np.isfinite(pred)
    obs, pred = obs[ok], pred[ok]
    if len(obs) < 2:
        return np.nan, np.nan
    return 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2), 100 * np.mean(np.abs((obs - pred) / obs))


def proximo(forma, c, l, x):
    # x_{n+1} a partir de x_n para uma família (vetorizado em x)
    col, S = FORMAS[forma]
    if col == "passo_W":
        return x + S * np.pi / (PAR.at[c, "beta"] * (1 + PAR.at[c, "eta"]))
    r = RAIZES[c][l]
    if len(r) < 2:
        return np.full(len(x), np.nan)
    k = np.clip(np.searchsorted(r, x), 1, len(r) - 1)
    k = np.where(np.abs(r[k - 1] - x) <= np.abs(r[k] - x), k - 1, k)
    out = np.full(len(x), np.nan)
    ok = k + 1 < len(r)
    out[ok] = x[ok] + S * (r[k[ok] + 1] - r[k[ok]])
    return out


def sequencia(forma, c, l, x0, n_lim):
    # x_{k+1} = f(x_k) a partir da primeira ressonância observada até passar de x_M = 1
    seq = [x0]
    while len(seq) < n_lim:
        nxt = proximo(forma, c, l, np.array([seq[-1]]))[0]
        if not (np.isfinite(nxt) and nxt > seq[-1] and nxt <= 1.0):
            break
        seq.append(nxt)
    return np.array(seq)


TESTE_FAM = FAM.loc[FAM["curva"].isin(GLOBAIS_TESTE) & FAM["curva"].isin(RAIZES)]


def avaliar(forma):
    col, S = FORMAS[forma]
    te = TR_TESTE
    obs, xn = te["x_np1"].to_numpy(float), te["x_n"].to_numpy(float)
    pred = xn + S * te[col].to_numpy(float)
    passo = obs - xn
    erro_passo = np.where(passo > 0, np.abs((pred - xn) - passo) / np.where(passo > 0, passo, 1), np.nan)
    erro_pos = np.abs(pred - obs) / obs
    peq = (te["zT"] < np.pi).to_numpy()
    out = {"x_n": xn, "obs": obs, "pred": pred, "passo_obs": passo, "passo_pred": pred - xn, "erro_passo": erro_passo,
           "erro_pos": erro_pos, "zT": te["zT"].to_numpy(float), "c": te["c_razao"].to_numpy(float)}
    for g, mk in (("pequena", peq), ("grande", ~peq), ("todas", np.ones_like(peq))):
        out[f"R2_{g}"], out[f"MAPE_{g}"] = r2_mape(obs[mk], pred[mk])
        e = erro_passo[mk][np.isfinite(erro_passo[mk])]
        out[f"passo_{g}"] = 100 * np.median(e) if len(e) else np.nan
        e = erro_pos[mk][np.isfinite(erro_pos[mk])]
        out[f"pos_{g}"] = 100 * np.median(e) if len(e) else np.nan
        out[f"n_{g}"] = int(mk.sum())
    desv, n_pred, n_obs = [], {}, {}
    for (c, l), d in TESTE_FAM.groupby(["curva", "l"]):
        o = np.sort(d["x"].to_numpy(float))
        s = sequencia(forma, c, l, o[0], 4 * len(o) + 50)
        n = min(len(o), len(s))
        if n >= 2:
            desv.append(np.abs(s[1:n] - o[1:n]) / o[1:n])
        n_pred[c] = n_pred.get(c, 0) + len(s)
        n_obs[c] = n_obs.get(c, 0) + len(o)
    desv = np.concatenate(desv) if desv else np.array([])
    out["seq_todas"] = 100 * np.median(desv) if len(desv) else np.nan
    cont = pd.DataFrame({"curva": list(n_pred), "N_pred": list(n_pred.values()), "N_obs": [n_obs[c] for c in n_pred]})
    cont["erro_N"] = 100 * np.abs(cont["N_pred"] - cont["N_obs"]) / cont["N_obs"]
    out["N_todas"] = float(cont["erro_N"].median()) if len(cont) else np.nan
    out["contagem"] = cont
    return out


AVAL = {f: avaliar(f) for f in FORMAS}
tabela = pd.DataFrame({NOMES_FORMA_TXT[f]: {c: a[c] for c in ("n_todas", "R2_todas", "MAPE_todas", "pos_todas", "passo_pequena",
                                                               "passo_grande", "passo_todas", "seq_todas", "N_todas")}
                       for f, a in AVAL.items()}).T
print(tabela.round(4).to_string())
tabela.to_csv(os.path.join(TAB_DIR, "tabela_metricas_recorrencia.csv"))

# gráfico da recorrência contra o banco: distância prevista x observada, cor = erro relativo mediano na posição
a = AVAL["car_S"]
fig, ax = plt.subplots(figsize=(5, 3.5))
n = hexbin_padrao(ax, a["passo_obs"], a["passo_pred"], C=100 * a["erro_pos"], rotulo_cor=f"Erro relativo mediano na posição ({PCT})")
estilizar(ax, r"$x_{M,n+1}-x_{M,n}$ observada", r"$x_{M,n+1}-x_{M,n}$ prevista")
adicionar_caixa(ax, rf"$S={fmt_pm(S_CAR, ERRO_S_CAR).strip('$')}$" + "\n" + f"${milhar(n)}$ transições de teste" + "\n"
                + rf"Erro mediano na posição $={dec(a['pos_todas'], 3) if a['pos_todas'] >= 0.01 else sci(a['pos_todas'])}${PCT}",
                x=0.035, ha="left")
fig.subplots_adjust(left=0.15, right=0.98, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_recorrencia")
plt.show()
plt.close(fig)

### 9.4 Comparação com as formas ajustadas

A mesma recorrência com $S=1$ (as raízes da Eq. 5.106 sem nenhum ajuste) e a forma com o espaçamento médio da Eq. (5.120),

$$
x_{M,n+1}=x_{M,n}+S_W\,\frac{\pi}{c_M/c_L+c_M/c_T},
$$

com $S_W$ ajustado do mesmo jeito (mediana por família e mediana das famílias), ficam como comparação. Todas são avaliadas nas mesmas
curvas de teste: $R^2$ e MAPE de $x_{M,n+1}$, erro relativo mediano da posição e do passo $x_{M,n+1}-x_{M,n}$, desvio da sequência gerada a
partir da primeira ressonância de cada família e erro no número de ressonâncias até $x_M=1$, separando as transições abaixo do primeiro
modo transversal ($z_T=(c_M/c_T)\,x_{M,n}<\pi$) das demais.

In [ ]:
rot = {"n_todas": "Transições", "R2_todas": r"$R^2$ de $x_{M,n+1}$", "MAPE_todas": f"MAPE de $x_{{M,n+1}}$ ({PCT})",
       "pos_todas": f"Erro mediano da posição ({PCT})", "passo_pequena": f"Erro do passo, $z_T<\\pi$ ({PCT})",
       "passo_grande": f"Erro do passo, $z_T\\geq\\pi$ ({PCT})", "seq_todas": f"Desvio da sequência ({PCT})",
       "N_todas": f"Erro em $N$ ({PCT})"}
fmt = lambda k, v: "--" if not np.isfinite(v) else (f"${milhar(v)}$" if k == "n_todas" else f"${dec(v, 4 if k == 'R2_todas' else 2)}$")
salvar_tabela_texto("tabela_comparacao_recorrencia", list(rot.values()),
                    [NOMES_FORMA[f].replace(", ", "\n") for f in FORMAS],
                    [[fmt(k, AVAL[f][k]) for f in FORMAS] for k in rot], tabela, figsize=(11.0, 4.4), first_col_width=0.34,
                    fontsize=13)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, grupo in zip(axes, ("pequena", "grande")):
    for f, est in ESTILO.items():
        a = AVAL[f]
        mk = a["zT"] < np.pi if grupo == "pequena" else a["zT"] >= np.pi
        e = np.sort(100 * a["erro_passo"][mk])
        e = e[np.isfinite(e) & (e > 0)]
        if len(e):
            ax.plot(e, np.arange(1, len(e) + 1) / len(e), label=NOMES_FORMA[f], **est)
    ax.set_xscale("log")
    ax.set_ylim(0, 1.02)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: dec(v, 2)))
    estilizar(ax, f"Erro relativo do passo ({PCT})", "Fração das transições" if grupo == "pequena" else None)
    adicionar_caixa(ax, r"$z_T<\pi$" if grupo == "pequena" else r"$z_T\geq\pi$", x=0.035, ha="left")
adicionar_legenda(axes[1], loc="lower right", fontsize=8)
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.16, top=0.98, wspace=0.08)
salvar_figura(fig, "grafico_erro_do_passo")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 3.5))
for f, est in ESTILO.items():
    a = AVAL[f]
    med = pd.DataFrame({"c": a["c"], "e": 100 * a["erro_passo"]}).dropna()
    if len(med):
        bins = np.logspace(np.log10(med["c"].min()) - 1e-9, np.log10(med["c"].max()) + 1e-9, 11)
        g = med.groupby(pd.cut(med["c"], bins), observed=True)["e"].median()
        ax.plot([np.sqrt(b.left * b.right) for b in g.index], g.to_numpy(), marker="o", markersize=3, label=NOMES_FORMA[f], **est)
ax.set_xscale("log"); ax.set_yscale("log")
estilizar(ax, r"$c_L/c_M$", f"Erro mediano do passo ({PCT})")
adicionar_legenda(ax, loc="upper right", fontsize=8)
fig.subplots_adjust(left=0.15, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_erro_contra_cL_cM")
plt.show()
plt.close(fig)

### Espaçamento médio por família

Para cada família $(\text{curva},\ell)$ das curvas contadas pelos polos, com pelo menos $6$ ressonâncias em $x_M\le1$, mede-se o
espaçamento médio $\overline{\Delta x}_M$ na metade superior da família e compara-se $\pi/\overline{\Delta x}_M$ com a densidade de
raízes da Eq. (5.120), $c_M/c_L+c_M/c_T$ ($\ell\ge1$) ou $c_M/c_L$ ($\ell=0$), nos três modelos (nos viscoelásticos, com as velocidades de
referência). O ajuste livre $\pi/\overline{\Delta x}_M=p_1\,c_M/c_L+p_2\,c_M/c_T+p_0$ é impresso para cada modelo, e o segundo gráfico
mostra a mesma grandeza na esfera elástica contra $c_M/c_L$ apenas (só o ramo longitudinal), com a linha em $c_M/c_L=100$.

In [ ]:
N_MIN_FAM = 6


def espacamento_familias(m):
    par = RESUMO[m].set_index("_indice")
    pol = set(par.index[par["modo_contagem"] == "polos_area"])
    pk = PICOS[m].loc[np.isfinite(PICOS[m]["l"]) & PICOS[m]["curva"].isin(pol)]
    linhas = []
    for (c, l), g in pk.groupby(["curva", "l"]):
        x = np.unique(np.round(np.sort(g["x"].to_numpy()), 12))
        if len(x) < N_MIN_FAM:
            continue
        dx = np.mean(np.diff(x[len(x) // 2:]))
        if dx > 0:
            a, b = par.at[c, "beta"], par.at[c, "beta"] * par.at[c, "eta"] if l >= 1 else 0.0
            linhas.append((c, int(l), len(x), a, b, a + b, np.pi / dx))
    return pd.DataFrame(linhas, columns=["curva", "l", "N", "a", "b", "Phi", "y"])


ESPAC = {m: espacamento_familias(m) for m in MODELOS}
for m, f in ESPAC.items():
    print(f"{NOME_MODELO[m]}: {len(f)} famílias com pelo menos {N_MIN_FAM} ressonâncias")
    for rot_l, sel in (("l = 0", f["l"] == 0), ("l >= 1", f["l"] >= 1)):
        s = f.loc[sel]
        if len(s) >= 5:
            A = np.c_[s["a"], s["b"], np.ones(len(s))]
            if rot_l == "l = 0":
                A[:, 1] = 0.0
            p = np.linalg.lstsq(A, s["y"].to_numpy(), rcond=None)[0]
            print(f"   {rot_l:7s} N={len(s):6d}  p1={p[0]:.4f}  p2={p[1]:.4f}  p0={p[2]:+.3f}  "
                  f"desvio mediano de pi/Dx em relação à Eq. (5.120) = {100 * np.median(np.abs(s['Phi'] / s['y'] - 1)):.2f} %")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, m in zip(axes, MODELOS):
    f = ESPAC[m]
    n = hexbin_padrao(ax, f["Phi"], f["y"], rotulo_cor="Famílias") if len(f) else 0
    estilizar(ax, r"$c_M/c_L+c_M/c_T$", r"$\pi/\overline{\Delta x}_M$")
    adicionar_caixa(ax, TEX_MODELO[m] + "\n" + f"${milhar(n)}$ famílias", x=0.035, ha="left")
fig.subplots_adjust(left=0.05, right=0.985, bottom=0.16, top=0.98, wspace=0.30)
salvar_figura(fig, "DeltaW_esfera_elastica")
plt.show()
plt.close(fig)

f = ESPAC["elastico"]
fig, ax = plt.subplots(figsize=(5, 3.5))
n = hexbin_padrao(ax, f["a"], f["y"], rotulo_cor="Famílias") if len(f) else 0
ax.axvline(100.0, color=COLORS[3], linestyle=LINESTYLES[3], linewidth=LINEWIDTHS[3], zorder=1)
estilizar(ax, r"$c_M/c_L$", r"$\pi/\overline{\Delta x}_M$")
adicionar_caixa(ax, "Elástico" + "\n" + f"${milhar(n)}$ famílias", x=0.035, ha="left")
fig.subplots_adjust(left=0.15, right=0.98, bottom=0.15, top=0.98)
salvar_figura(fig, "DeltaW_elastico")
plt.show()
plt.close(fig)

## 10. Custo computacional

Tempo do cálculo completo por configuração (`runtime_s` do gerador, na máquina onde o banco de dados rodou) contra o tempo das relações
medido aqui: a relação de existência para todas as configurações de uma vez e, para cada curva de teste, as raízes da Eq. (5.106) de
todos os seus $\ell$ e a sequência de ressonâncias de cada família até $x_M=1$.

In [ ]:
resumo = RESUMO["elastico"]
t0 = time.perf_counter()
for _ in range(5):
    p_res(resumo["u"].to_numpy(), resumo["v"].to_numpy(), resumo["w"].to_numpy(), COEF_EXIST)
t_exist = (time.perf_counter() - t0) / 5 / len(resumo)

custo = []
for c, d in TESTE_FAM.groupby("curva"):
    beta, eta, rho = PAR.at[c, "beta"], PAR.at[c, "eta"], PAR.at[c, "rho_ratio"]
    t0 = time.perf_counter()
    R = raizes_caracteristica(rho, eta, Z_FATOR * beta, sorted(d["l"].unique()))
    for l, dl in d.groupby("l"):
        sequencia("car_S", c, l, dl["x"].min(), 4 * len(dl) + 50)
    custo.append({"c": PAR.at[c, "c_razao"], "completo": PAR.at[c, "runtime_s"], "relacoes": time.perf_counter() - t0 + t_exist})
custo = pd.DataFrame(custo)
print(f"existência: {t_exist:.2g} s por configuração | cálculo completo: mediana {resumo['runtime_s'].median():.3g} s")
if len(custo):
    print(f"relações: mediana {custo['relacoes'].median():.3g} s | ganho mediano {np.median(custo['completo'] / custo['relacoes']):.3g}x")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(resumo["c_razao"], resumo["runtime_s"], s=8, facecolors="white", edgecolors="0.45", linewidths=0.5,
           label="Cálculo completo", zorder=2)
if len(custo):
    ax.scatter(custo["c"], custo["relacoes"], s=10, facecolors="#8B1A1A", edgecolors="black", linewidths=0.3,
               label="Relações", zorder=3)
ax.axhline(t_exist, color="black", linestyle=":", linewidth=1.0, label="Só a existência")
ax.set_xscale("log"); ax.set_yscale("log")
estilizar(ax, r"$c_L/c_M$", "Tempo por configuração (s)")
adicionar_legenda(ax, loc="lower left", bbox_to_anchor=(0.02, 0.12))
fig.subplots_adjust(left=0.16, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_custo")
plt.show()
plt.close(fig)

## 11. Arquivos gerados

Lista o que foi gravado em `resultados/` e cria `resultados_esfera_viscoelastica.zip` ao lado da pasta (no Colab, dentro de
`Meu Drive/visco_final/`), pronto para baixar.

In [ ]:
for sub in (FIG_DIR, TAB_DIR):
    arqs = sorted(os.listdir(sub))
    nomes = sorted({os.path.splitext(a)[0] for a in arqs})
    print(f"{sub}: {len(arqs)} arquivos, {len(nomes)} itens")
    for n in nomes:
        print("   ", n)
zip_path = shutil.make_archive(os.path.join(os.path.dirname(SAIDA), "resultados_esfera_viscoelastica"), "zip",
                               root_dir=os.path.dirname(SAIDA), base_dir=os.path.basename(SAIDA))
print("zip:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if NO_COLAB:
    print("No Colab os arquivos já estão no Drive; a sincronização pode levar alguns minutos para aparecer no navegador.")